# Image Captioning on the Flickr8k Dataset

Implementation and evaluation of five image captioning architectures trained on the
[Flickr8k dataset](https://www.kaggle.com/datasets/adityajn105/flickr8k).

## Models

- **Show and Tell** - CNN encoder (ResNet-18) + LSTM decoder with transfer learning
- **Show, Attend and Tell** - additive spatial attention over CNN feature maps
- **Beam search decoding** - comparison against greedy decoding on BLEU scores
- **Attention visualization** - per-word attention maps over image regions
- **GloVe embeddings** - pretrained word vectors vs. random initialization
- **Dot-product attention** - scaled dot-product vs. additive Bahdanau attention
- **Transformer decoder** - cross-attention to spatial image features

## Setup

Requires the Flickr8k dataset placed under `data/` and the helper modules
`dataloader.py` and `tokenizer.py` in the same directory.
Weights & Biases logging is optional — see the import cell below.

## Note on validation split

Flickr8k's official validation set was used as a training proxy for early stopping
and checkpointing. The reported test metrics therefore carry a slight optimistic bias,
as the checkpoint selection criterion overlapped with the evaluation set. A
methodologically cleaner setup would require the validation set to be held out
entirely from checkpoint selection.

## Project Setup

In [ ]:
import math
import textwrap

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.models as models
from torchvision import transforms
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
import zipfile, os
from nltk.translate.meteor_score import meteor_score
import urllib.request

nltk.download('punkt',      quiet=True)
nltk.download('punkt_tab',  quiet=True)
nltk.download('wordnet',    quiet=True)

from dataloader import build_tokenizer_from_split, create_caption_dataloader
from tokenizer import tokenize

# Weights & Biases is optional. Set USE_WANDB = False to skip logging entirely.
USE_WANDB = None
try:
    import wandb
    os.environ['WANDB_SILENT'] = 'true'
    if USE_WANDB:
        wandb.login()
        WANDB_PROJECT = 'image-captioning-flickr8k'
        WANDB_ENTITY  = None  # set to your W&B username or team
except ImportError:
    USE_WANDB = False
    print('wandb not installed - logging disabled')

print('torch version:', torch.__version__)
print('torchvision version:', torchvision.__version__)

## Image Preprocessing Pipeline

1. **Constants** → ImageNet statistics (size, mean, std)
2. **RGB Conversion** → Ensure consistent 3-channel input
3. **Train Transforms** → Augmentation + Normalization (training)
4. **Test Transforms** → Clean resize + Normalization (inference)
5. **Denormalize** → Reverse normalization for visualization

---
*Core idea: Prepare raw images into normalized tensors that a pretrained CNN expects, with randomized augmentations at training time to improve generalization.*

In [ ]:
# ── 1. Constants ──────────────────────────────────────────────────────────────
# Most pretrained models (ResNet, ViT, etc.) were trained on ImageNet with these
# exact statistics. If you use different values the model receives a shifted input
# distribution and the pretrained weights will no longer work as expected.
IMAGENET_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# ── 2. RGB Conversion ─────────────────────────────────────────────────────────
# Images can come in many modes: RGBA (with transparency), L (grayscale) or
# P (palette-based). A CNN with 3 input channels will crash or produce garbage
# if it receives a 1-channel or 4-channel image, so we enforce RGB everywhere.
def _convert_to_rgb(image: Image.Image) -> Image.Image:
    return image.convert('RGB')

# ── 3. Train Transforms ───────────────────────────────────────────────────────
def build_train_transforms() -> transforms.Compose:
    return transforms.Compose([
        transforms.Lambda(_convert_to_rgb),

        # Randomly crops and resizes to 224x224. The model learns to recognize
        # objects even when they are partially visible or at different scales,
        # which makes it more robust on real-world images.
        transforms.RandomResizedCrop(IMAGENET_SIZE, scale=(0.75, 1.0)),

        # Most objects look identical when mirrored so flipping effectively
        # doubles the training set at no extra cost. Do not use this for tasks
        # where orientation matters such as reading text or medical scans.
        transforms.RandomHorizontalFlip(),

        # Randomly shifts brightness, contrast, saturation and hue so the model
        # does not overfit to the specific lighting conditions in the training set.
        # Hue is kept small because large shifts change object colors completely.
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),

        # Converts a PIL image with values in [0, 255] to a float tensor in [0, 1]
        # and reorders the axes from (H, W, C) to (C, H, W) as PyTorch expects.
        # This must come before Normalize because Normalize expects a float tensor.
        transforms.ToTensor(),

        # Shifts the input distribution to have zero mean and unit variance per channel.
        # This keeps gradient magnitudes balanced across layers and makes training
        # significantly faster and more stable.
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

# ── 4. Test Transforms ────────────────────────────────────────────────────────
# No augmentation at test time because we want deterministic and reproducible
# predictions. Random crops or flips would give different results on every run
# which makes it impossible to fairly compare model checkpoints.
def build_test_transforms() -> transforms.Compose:
    return transforms.Compose([
        transforms.Lambda(_convert_to_rgb),

        # A fixed resize instead of a random crop ensures the full object stays
        # visible. The simple alternative of Resize(256) followed by CenterCrop(224)
        # preserves the aspect ratio better but is slightly more complex.
        transforms.Resize((IMAGENET_SIZE, IMAGENET_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

# ── 5. Denormalize ────────────────────────────────────────────────────────────
# Normalization shifts pixel values outside the [0, 1] range that matplotlib
# expects for display. This function applies the inverse formula to recover
# the original pixel values so the image can be visualized correctly.
# new_tensor() is used instead of torch.tensor() because it automatically places
# the result on the same device and dtype as the input image.
def denormalize_image(image: torch.Tensor) -> torch.Tensor:
    mean_t = image.new_tensor(IMAGENET_MEAN).view(-1, 1, 1)  # Shape: (3, 1, 1)
    std_t  = image.new_tensor(IMAGENET_STD).view(-1, 1, 1)   # Shape: (3, 1, 1)
    # view(-1, 1, 1) reshapes to (3, 1, 1) so the values broadcast correctly
    # over all pixels in the (3, H, W) image tensor without explicit loops
    return (image * std_t + mean_t).clamp(0, 1)

## Image Captioning, Data Setup & Loading

1. **Hyperparameters** → Global constants controlling vocab & batching
2. **Tokenizer** → Build word → index vocabulary from training text
3. **DataLoaders** → Pair images with captions, apply transforms, batch
4. **Special Tokens** → Extract PAD / BOS / EOS indices for the model
5. **Dataset Statistics** → Images, captions, and avg caption length per split
6. **Sanity Check** → Peek at one batch, print shapes and dataset sizes

---
*Core idea: Convert raw image files + text captions into paired, numerically encoded, fixed-length batches ready for a CNN+RNN/Transformer.*

In [ ]:
# ── 1. Hyperparameters ────────────────────────────────────────────────────────
data_dir = 'data/'

# Words below this frequency threshold are replaced by <UNK> at training time
# which forces the model to handle unknown words rather than memorizing rare ones
# that will likely never appear at test time
MIN_WORD_FREQ = 5

# All captions in a batch must have the same length because GPUs process data
# in parallel as fixed-size matrices. Longer captions get truncated, shorter
# ones get filled with PAD tokens up to this limit.
MAX_LEN    = 40

BATCH_SIZE = 32

# ── 2. Tokenizer ──────────────────────────────────────────────────────────────
# Building the vocabulary only from training data is critical because the
# vocabulary defines what the model can output. If test words were included,
# the model would have an unfair advantage on words it should treat as unknown.
tokenizer = build_tokenizer_from_split(
    split='train', data_dir=data_dir, min_freq=MIN_WORD_FREQ
)

# ── 3. DataLoaders ────────────────────────────────────────────────────────────
# caption_sampling='all' uses all ~5 reference captions per image as separate
# training examples. This is important because different captions describe the
# same image in different ways, teaching the model that there is no single
# correct caption for a given image.
train_loader = create_caption_dataloader(
    split='train', data_dir=data_dir, tokenizer=tokenizer,
    transform=build_train_transforms(), batch_size=BATCH_SIZE,
    max_len=MAX_LEN, caption_sampling='all',
)

# caption_sampling='first' always picks the first caption per image at test time.
# This makes perplexity evaluation fully deterministic across runs, unlike 'random',
# which draws a different caption each time and causes small PPL fluctuations.
test_loader = create_caption_dataloader(
    split='test', data_dir=data_dir, tokenizer=tokenizer,
    transform=build_test_transforms(), batch_size=BATCH_SIZE,
    max_len=MAX_LEN, caption_sampling='first',
)

# ── 4. Special Tokens ─────────────────────────────────────────────────────────
# These indices are used throughout training and inference so we extract them once here.
# PAD_IDX is passed to CrossEntropyLoss as ignore_index so padded positions
# do not contribute to the gradient and the model is not penalized for them.
# BOS_IDX is the first token fed to the decoder at every generation step.
# EOS_IDX is what the model must learn to emit when the caption is complete.
VOCAB_SIZE = len(tokenizer)
PAD_IDX    = tokenizer.pad_idx
BOS_IDX    = tokenizer.bos_idx
EOS_IDX    = tokenizer.eos_idx

# ── 5. Dataset Statistics ─────────────────────────────────────────────────────
# Summarising the dataset before training gives context for interpreting results:
# the small size (6k train images vs. MS-COCO's 120k) explains the limited BLEU
# scores and makes regularisation and augmentation especially important.
train_imgs   = len(train_loader.dataset.captions_map)
test_imgs    = len(test_loader.dataset.captions_map)
train_caps   = len(train_loader.dataset.samples)
test_caps    = sum(len(v) for v in test_loader.dataset.captions_map.values())
caps_per_img = train_caps / train_imgs
avg_len      = sum(
    len(cap.split())
    for caps in train_loader.dataset.captions_map.values()
    for cap in caps
) / train_caps

print(f'Vocabulary size:           {VOCAB_SIZE:,}')
print(f'')
print(f'{"Split":<10} {"Images":>8} {"Captions":>10} {"Caps/Image":>11} {"Avg Length":>11}')
print(f'{"-"*44}')
print(f'{"Train":<10} {train_imgs:>8,} {train_caps:>10,} {caps_per_img:>11.1f} {avg_len:>10.1f}w')
print(f'{"Test":<10} {test_imgs:>8,} {test_caps:>10,} {test_caps/test_imgs:>11.1f}')
print(f'')
print(f'Note: BLEU compares against all {int(test_caps/test_imgs):.0f} test references per image.')
print(f'      Perplexity uses 1 caption per image (caption_sampling=first).')

# ── 6. Sanity Check ───────────────────────────────────────────────────────────
# Fetching one batch before training starts catches broken transforms or
# mismatched shapes immediately rather than after hours of training.
# images:        Shape (B, 3, 224, 224), normalized image tensors
# captions:      Shape (B, MAX_LEN), each row is one padded token sequence
# lengths:       Shape (B,), actual caption length before padding, needed to
#                ignore PAD tokens in RNN-based models via pack_padded_sequence
# image_ids:     original dataset IDs used to look up reference captions for BLEU
# raw_captions:  original text strings for debugging tokenization errors
images, captions, lengths, image_ids, raw_captions = next(iter(train_loader))

print(f'')
print(f'Batch image shape:         {tuple(images.shape)}')
print(f'Batch caption shape:       {tuple(captions.shape)}')

## Caption Visualization

1. **Sample Selection** → Limit display to n examples
2. **Image Decoding** → Reverse normalization for display
3. **Caption Decoding** → Convert token indices back to words
4. **Plot Layout** → Show image and caption side by side

---
*Core idea: Verify that images and captions are correctly paired and that the tokenizer encodes and decodes without information loss.*

In [ ]:
# ── 1. Sample Selection ───────────────────────────────────────────────────────
# min() protects against the last batch being smaller than num_examples
# because PyTorch does not guarantee that every batch has exactly BATCH_SIZE samples
num_examples = 6
n = min(num_examples, len(images))

# ── 2. Image Decoding ─────────────────────────────────────────────────────────
# width_ratios=[1, 2] gives the caption column twice the space of the image column
# so the text does not get cut off
fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n),
                         gridspec_kw={'width_ratios': [1, 2]})
for idx in range(n):
    # detach() is required because numpy cannot handle tensors that are still
    # part of the computation graph. .cpu() moves the tensor from GPU to RAM first.
    # permute(1, 2, 0) reorders from (C, H, W) to (H, W, C) because PyTorch stores
    # images channel-first but matplotlib expects channel-last
    img = denormalize_image(images[idx].detach().cpu()).permute(1, 2, 0).numpy()

    # ── 3. Caption Decoding ───────────────────────────────────────────────────
    # lengths[idx] tells us where the real caption ends and padding begins
    # so we slice only the meaningful tokens before decoding
    encoded = captions[idx, :int(lengths[idx])].tolist()

    # skip_special_tokens=False keeps BOS and EOS visible in the output
    # so we can verify that the tokenizer correctly wraps every caption
    decoded = tokenizer.decode(encoded, skip_special_tokens=False)

    # ── 4. Plot Layout ────────────────────────────────────────────────────────
    axes[idx][0].imshow(img)
    axes[idx][0].set_title(image_ids[idx], fontsize=8)
    axes[idx][0].axis('off')

    # Showing all three representations (raw, encoded, decoded) side by side
    # lets you immediately spot tokenization errors or mismatched image-caption pairs
    # textwrap.fill breaks long strings at 70 characters so they fit the column
    axes[idx][1].axis('off')
    axes[idx][1].text(0, 1,
        f'Raw:     {textwrap.fill(raw_captions[idx], 70)}\n\n'
        f'Encoded: {textwrap.fill(str(encoded), 70)}\n\n'
        f'Decoded: {textwrap.fill(decoded, 70)}',
        va='top', ha='left', fontsize=9, family='monospace')

plt.tight_layout()
plt.show()

## Shared Utilities

Common training loop, loss, BLEU/perplexity computation and visualisation helpers shared across all experiments.

In [ ]:
# Check if a GPU is available and use it if possible. 
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('device:', DEVICE)

## Training and Evaluation Pipeline

1. **train_one_epoch** → One full pass over the training set with gradient updates
2. **eval_perplexity** → Measure conditional token likelihood on a chosen split
3. **eval_bleu** → Measure caption quality by comparing generated text to references
4. **train_model** → Orchestrate epochs, checkpointing and learning rate scheduling
5. **plot_curves** → Visualize loss and perplexity over training
6. **show_generated_captions** → Visually inspect generated captions next to images
7. **generate_beam** → Greedy decoding replaced by beam search for better caption quality

---
*Core idea: Train the model to predict the next word given an image and the previous words, then evaluate whether the generated captions are fluent and accurate.*

In [ ]:
# ── 1. train_one_epoch ────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, device):
    # model.train() activates Dropout and BatchNorm in training mode.
    # Without this call after an eval phase, the model would behave as if
    # Dropout were turned off and gradients would be unreliable.
    model.train()
    total_loss, n_batches = 0.0, 0

    for imgs, caps, _, _, _ in loader:
        # Move data to GPU. This must happen before every forward pass because
        # PyTorch does not automatically transfer tensors between devices.
        imgs, caps = imgs.to(device), caps.to(device)

        # 1) Zero Grad: PyTorch accumulates gradients by default so we must
        # reset them at the start of every batch to avoid incorrect updates.
        optimizer.zero_grad()

        # 2) Forward Pass
        # caps is passed so the model can use teacher forcing: at each step
        # the model receives the ground truth previous token instead of its
        # own prediction. This makes training faster and more stable.
        logits = model(imgs, caps)   # Shape: (B, T-1, V)

        # The target is caps shifted left by one position because the model
        # must predict the next token. caps[:, 0] is BOS which is never a
        # target, so we skip it. caps[:, 1:] starts at the first real word.
        targets = caps[:, 1:]        # Shape: (B, T-1)

        # 3) Loss: reshape flattens batch and time into one dimension because
        # cross_entropy expects (N, V) logits and (N,) targets.
        # ignore_index=PAD_IDX ensures padded positions do not contribute
        # to the loss so the model is not penalized for predicting padding.
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),   # Shape: (B*(T-1), V)
            targets.reshape(-1),                    # Shape: (B*(T-1),)
            ignore_index=PAD_IDX,
        )

        # 4) Backward: compute gradients for all parameters via backpropagation
        loss.backward()

        # Gradient clipping prevents exploding gradients by rescaling the
        # gradient vector to have a maximum norm of 5. This is especially
        # important in RNNs where gradients can grow very large over long sequences.
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        # 5) Optimizer Step: update all parameters using the computed gradients
        optimizer.step()

        # loss.item() detaches the scalar from the computation graph so it
        # does not keep the entire batch in memory across iterations
        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches


# ── 2. eval_perplexity ────────────────────────────────────────────────────────
def eval_perplexity(model, loader, device):
    # model.eval() disables Dropout and switches BatchNorm to use running
    # statistics instead of batch statistics, giving deterministic predictions
    model.eval()
    total_loss, total_tokens = 0.0, 0

    # torch.no_grad() disables gradient tracking during evaluation.
    # This saves memory and speeds up inference because no computation
    # graph is built for the backward pass.
    with torch.no_grad():
        for imgs, caps, lengths, _, _ in loader:
            imgs, caps = imgs.to(device), caps.to(device)
            logits  = model(imgs, caps)
            targets = caps[:, 1:]

            # mask identifies which positions are real tokens rather than padding.
            # We need the exact count of real tokens to compute perplexity correctly.
            mask = targets != PAD_IDX

            # reduction='sum' returns the total loss across all tokens instead
            # of the mean. This is necessary because we want to average over
            # all real tokens globally across the entire dataset, not per batch.
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=PAD_IDX,
                reduction='sum',
            )
            total_loss   += loss.item()
            total_tokens += mask.sum().item()

    # Perplexity is defined as exp(average negative log-likelihood per token).
    # A perplexity of 10 means the model is as uncertain as choosing uniformly
    # among 10 equally likely words at every step. Lower is better.
    return math.exp(total_loss / total_tokens)


# ── 3. eval_bleu ──────────────────────────────────────────────────────────────
def eval_bleu(model, loader, tokenizer, device,
              is_attention=False, max_len=35):
    model.eval()
    references, hypotheses = [], []

    # captions_map maps each image ID to all its reference captions.
    # BLEU compares the generated caption against all references so having
    # multiple references gives a fairer score than using just one.
    captions_map = loader.dataset.captions_map

    with torch.no_grad():
        for imgs, _, _, img_ids, _ in loader:
            imgs = imgs.to(device)

            # Attention models return both the generated tokens and the attention
            # weights, so we unpack differently depending on the model type
            if is_attention:
                gen, _ = model.generate(imgs, max_len=max_len,
                                        bos_idx=tokenizer.bos_idx,
                                        eos_idx=tokenizer.eos_idx)
            else:
                gen = model.generate(imgs, max_len=max_len,
                                     bos_idx=tokenizer.bos_idx,
                                     eos_idx=tokenizer.eos_idx)

            for i, img_id in enumerate(img_ids):
                # Each reference caption is tokenized into a list of words.
                # BLEU operates on word lists, not raw strings.
                refs = [tokenize(c) for c in captions_map[img_id]]
                references.append(refs)

                toks = gen[i].cpu().tolist()

                # Truncate at the EOS token because everything after it is
                # meaningless filler that would unfairly lower the BLEU score
                if tokenizer.eos_idx in toks:
                    toks = toks[:toks.index(tokenizer.eos_idx)]

                hyp = tokenizer.decode(toks, skip_special_tokens=True).split()
                hypotheses.append(hyp)

    # METEOR considers synonym matches via WordNet, which makes it more
    # robust than BLEU for paraphrased but semantically correct captions
    meteor = sum(
    meteor_score(refs, hyp)
    for refs, hyp in zip(references, hypotheses)
    ) / len(hypotheses)

    # Smoothing is applied because short captions often have zero n-gram matches
    # for higher orders like BLEU-3 and BLEU-4, which would give a score of 0
    # even for reasonable captions. method1 adds a small count to avoid this.
    sf = SmoothingFunction().method1
    return {
        # The weight tuples define which n-gram orders to use.
        # BLEU-1 uses only unigrams, BLEU-4 averages over 1 to 4-grams.
        'BLEU-1': corpus_bleu(references, hypotheses, (1,0,0,0),       smoothing_function=sf),
        'BLEU-2': corpus_bleu(references, hypotheses, (.5,.5,0,0),     smoothing_function=sf),
        'BLEU-3': corpus_bleu(references, hypotheses, (1/3,1/3,1/3,0), smoothing_function=sf),
        'BLEU-4': corpus_bleu(references, hypotheses, (.25,.25,.25,.25),smoothing_function=sf),
        'METEOR': meteor,
    }


# ── 4. train_model ────────────────────────────────────────────────────────────
def train_model(model, train_loader, test_loader, optimizer, scheduler,
                device, num_epochs=20, model_name='model',
                is_attention=False, early_stopping_patience=None, config=None):
    
    wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=model_name, config=config or {})
    
    train_losses, val_ppls = [], []
    best_ppl = float('inf')
    patience_counter = 0

    for epoch in range(1, num_epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, device)
        ppl  = eval_perplexity(model, test_loader, device)
        wandb.log({"train_loss": loss, "val_perplexity": ppl, "epoch": epoch})
        train_losses.append(loss)
        val_ppls.append(ppl)

        # ReduceLROnPlateau reduces the learning rate when perplexity stops
        # improving. Passing ppl here lets the scheduler track validation
        # performance rather than training loss, which is more meaningful.
        if scheduler is not None:
            scheduler.step(ppl)

        # Save the model weights only when validation perplexity improves.
        # This way the best checkpoint is preserved even if later epochs overfit.
        if ppl < best_ppl:
            best_ppl = ppl
            patience_counter = 0
            torch.save(model.state_dict(), f'best_{model_name}.pth')
        else:
            patience_counter += 1

        print(f'Epoch {epoch:3d}/{num_epochs}  loss={loss:.4f}  val_ppl={ppl:.2f}')

        # Early stopping halts training when val_ppl has not improved for
        # early_stopping_patience consecutive epochs. The best checkpoint
        # saved above is unaffected, so no progress is lost.
        if early_stopping_patience and patience_counter >= early_stopping_patience:
            print(f'Early stopping at epoch {epoch} (no improvement for {early_stopping_patience} epochs)')
            break

    wandb.finish()
    return train_losses, val_ppls



# ── 5. plot_curves ────────────────────────────────────────────────────────────
def plot_curves(losses, ppls, title=''):
    best_epoch = ppls.index(min(ppls)) + 1

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)

    ax1.plot(losses, color='steelblue', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Train Loss')
    ax1.set_title('Training Loss')
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    ax2.plot(ppls, color='darkorange', linewidth=2)
    ax2.axvline(x=best_epoch - 1, color='green', linestyle='--',
                linewidth=1.5, label=f'Best epoch {best_epoch}')
    ax2.annotate(f'PPL {min(ppls):.2f}',
                 xy=(best_epoch - 1, min(ppls)),
                 xytext=(best_epoch + 1, min(ppls) + 1),
                 arrowprops=dict(arrowstyle='->', color='green'),
                 color='green', fontsize=9)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Val Perplexity')
    ax2.set_title('Validation Perplexity')
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.legend()

    plt.tight_layout()
    plt.show()


# ── 6. show_generated_captions ────────────────────────────────────────────────
def show_generated_captions(model, loader, tokenizer, device,
                             n=6, is_attention=False, title=''):
    model.eval()
    imgs, caps, lengths, img_ids, raw_caps = next(iter(loader))

    # Only pass the first n images to the model to keep the output manageable
    imgs = imgs[:n].to(device)

    with torch.no_grad():
        if is_attention:
            gen, _ = model.generate(imgs, max_len=35,
                                    bos_idx=tokenizer.bos_idx,
                                    eos_idx=tokenizer.eos_idx)
        else:
            gen = model.generate(imgs, max_len=35,
                                  bos_idx=tokenizer.bos_idx,
                                  eos_idx=tokenizer.eos_idx)

    fig, axes = plt.subplots(n, 1, figsize=(10, 3 * n))
    fig.suptitle(title, fontsize=12)

    for i in range(n):
        img = denormalize_image(imgs[i].cpu()).permute(1, 2, 0).numpy()

        toks = gen[i].cpu().tolist()
        # Stop at EOS so the displayed caption is clean and does not include
        # the padding tokens that follow the end of sequence marker
        if tokenizer.eos_idx in toks:
            toks = toks[:toks.index(tokenizer.eos_idx)]

        cap = tokenizer.decode(toks, skip_special_tokens=True)

        # Showing GEN and REF together makes it easy to judge caption quality
        # without having to look up the reference separately
        axes[i].imshow(img)
        axes[i].set_title(f'GEN: {cap}\nREF: {raw_caps[i]}', fontsize=9)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# ── 7. generate_beam ──────────────────────────────────────────────────────────
def generate_beam(model, images, tokenizer, beam_size=5, max_len=35):
    model.eval()
    results = []
    with torch.no_grad():
        for i in range(images.size(0)):
            img = images[i].unsqueeze(0)             # Shape: (1, 3, H, W)
            features = model.encoder(img)
            h, c = model._init_hidden(features)

            # Each beam tracks its cumulative log-probability score, the tokens
            # generated so far, and the current LSTM hidden and cell state
            beams = [(0.0, [tokenizer.bos_idx], h, c)]

            for _ in range(max_len):
                candidates = []
                for score, tokens, h, c in beams:
                    # If this beam already produced EOS we keep it unchanged
                    # because it has finished generating a complete sentence
                    if tokens[-1] == tokenizer.eos_idx:
                        candidates.append((score, tokens, h, c))
                        continue
                    word = torch.tensor([tokens[-1]], device=images.device)
                    emb = model.embed(word)            # Shape: (1, embed_dim)
                    h, c = model.lstm(emb, (h, c))

                    # log_softmax converts raw scores into log-probabilities so
                    # we can add them across steps instead of multiplying, which
                    # avoids numerical underflow for long sequences
                    log_probs = torch.log_softmax(model.fc(h), dim=-1)
                    topk_probs, topk_idx = log_probs.topk(beam_size)
                    for prob, idx in zip(topk_probs[0], topk_idx[0]):
                        candidates.append((score + prob.item(), tokens + [idx.item()], h, c))

                # Keep only the top beam_size candidates by cumulative score
                beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_size]

            # The highest scoring beam is the final generated caption
            best_tokens = beams[0][1]
            if tokenizer.eos_idx in best_tokens:
                best_tokens = best_tokens[:best_tokens.index(tokenizer.eos_idx)]

            # Skip the BOS token at position 0 before decoding to a string
            results.append(tokenizer.decode(best_tokens[1:], skip_special_tokens=True))
    return results

## Model 1: Show and Tell

Encoder-decoder captioning model from [Vinyals et al., 2015](https://arxiv.org/abs/1411.4555).  
- **Encoder**: pretrained ResNet-18 (global average-pooled to a 512-dim vector).  
- **Decoder**: single-layer LSTM, image feature initialises the hidden state.  
- **Training**: teacher forcing, greedy decoding at inference.

In [ ]:
# --- Hyperparameters ---
SAT_ENCODER_DIM  = 512
SAT_EMBED_DIM    = 256
SAT_DECODER_DIM  = 512

# 0.5 is the standard starting point for caption models because the decoder
# tends to overfit quickly on the limited vocabulary of image descriptions
SAT_DROPOUT      = 0.5

# Freezing the ResNet body means only the decoder gets trained in the first
# phase. This prevents the pretrained visual features from being destroyed
# before the decoder has learned anything useful.
SAT_FREEZE       = True

# 3e-4 is the commonly recommended default for Adam, known as
# "Karpathy's constant" in the community
SAT_LR           = 3e-4

SAT_EPOCHS       = 40
SAT_WD           = 1e-4

# 30x smaller than SAT_LR to avoid catastrophic forgetting: the backbone
# weights are already well-trained and only need minor adjustments
SAT_FINETUNE_LR     = 1e-5

# Fewer epochs than pretraining because the decoder is already stable
# and the encoder only needs small corrections at this stage
SAT_FINETUNE_EPOCHS = 10

## CNN Encoder and LSTM Decoder using Transfer Learning

1. **ShowAndTellEncoder** → Extract a single feature vector from the image using pretrained ResNet-18
2. **ShowAndTellCaptioner** → Wire up all layers of the encoder-decoder model
3. **_init_hidden** → Project image features into the LSTM starting state
4. **forward** → Teacher-forced training pass through the LSTM
5. **generate** → Greedy decoding at inference time

---
*Core idea: Reuse a ResNet-18 pretrained on ImageNet to extract visual features, compress them into one vector, use it to initialize an LSTM, then let the LSTM generate one word at a time conditioned on that visual context.*

In [ ]:
# ── 1. ShowAndTellEncoder ─────────────────────────────────────────────────────
class ShowAndTellEncoder(nn.Module):
    """ResNet-18 backbone producing a single global feature vector."""

    def __init__(self, encoder_dim: int = 512, freeze: bool = True):
        super().__init__()

        # Transfer Learning: ResNet-18 was pretrained on ImageNet and has already
        # learned to recognize edges, textures and objects. We reuse this knowledge
        # because our caption dataset is too small to learn visual features from scratch.
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # Remove the last two layers because they were trained to classify ImageNet
        # categories. We only want the visual features beneath them.
        self.body = nn.Sequential(*list(resnet.children())[:-2])
        # Shape after body: (B, 512, H', W') where H' and W' depend on input size

        # AdaptiveAvgPool2d collapses any spatial resolution down to (1, 1)
        # so the encoder works with any image size and always produces a fixed-length vector
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.encoder_dim = encoder_dim

        # Freezing the backbone protects the pretrained weights from being
        # overwritten early in training before the decoder has learned anything.
        # Once the decoder is stable, you can unfreeze for fine-tuning.
        if freeze:
            for p in self.body.parameters():
                p.requires_grad_(False)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.body(images)      # Shape: (B, 512, H', W')
        features = self.pool(features)    # Shape: (B, 512, 1, 1)
        return features.flatten(1)        # Shape: (B, 512)


# ── 2. ShowAndTellCaptioner.__init__ ──────────────────────────────────────────
class ShowAndTellCaptioner(nn.Module):
    """Show and Tell: CNN encoder + LSTM decoder with teacher forcing."""

    def __init__(
        self,
        vocab_size: int,
        encoder_dim: int = 512,
        embed_dim:   int = 256,
        decoder_dim: int = 512,
        dropout:     float = 0.5,
        freeze_encoder: bool = True,
    ):
        super().__init__()
        self.encoder = ShowAndTellEncoder(encoder_dim, freeze_encoder)

        # padding_idx=PAD_IDX tells the embedding layer to always return a zero
        # vector for PAD tokens and to skip them during gradient computation
        self.embed   = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # These two linear layers project the image vector into the initial
        # hidden state (h) and cell state (c) of the LSTM. This is how the
        # visual information enters the decoder at the very first time step.
        self.init_h  = nn.Linear(encoder_dim, decoder_dim)
        self.init_c  = nn.Linear(encoder_dim, decoder_dim)

        # LSTMCell processes one time step at a time, giving us full control
        # over the loop. This is necessary because at inference time we need
        # to feed the previous output back as the next input.
        self.lstm    = nn.LSTMCell(embed_dim, decoder_dim)
        self.dropout = nn.Dropout(dropout)

        # Project the LSTM hidden state onto a distribution over the vocabulary
        self.fc      = nn.Linear(decoder_dim, vocab_size)

    # ── 3. _init_hidden ───────────────────────────────────────────────────────
    def _init_hidden(self, features):
        # tanh squashes the projected features into [-1, 1], which is the
        # expected value range for LSTM hidden and cell states
        h = torch.tanh(self.init_h(features))   # Shape: (B, decoder_dim)
        c = torch.tanh(self.init_c(features))   # Shape: (B, decoder_dim)
        return h, c

    # ── 4. forward ────────────────────────────────────────────────────────────
    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        """Teacher-forced forward pass.

        Args:
            images:   (B, 3, H, W)
            captions: (B, T)  — includes BOS and EOS/PAD
        Returns:
            logits: (B, T-1, vocab_size)  — predictions for positions 1..T
        """
        features = self.encoder(images)            # Shape: (B, encoder_dim)
        h, c = self._init_hidden(features)

        # Drop the last token because we never need to predict what comes after
        # EOS. The model sees BOS and every word up to the second-to-last token
        # and must predict the next word at each step.
        embeddings = self.embed(captions[:, :-1])  # Shape: (B, T-1, embed_dim)

        logits = []
        for t in range(embeddings.size(1)):
            # Teacher forcing: feed the ground truth word at step t instead of
            # the word the model predicted. This stabilizes training because
            # early prediction errors do not cascade through the whole sequence.
            h, c = self.lstm(self.dropout(embeddings[:, t]), (h, c))
            logits.append(self.fc(self.dropout(h)))  # each entry: (B, vocab_size)

        return torch.stack(logits, dim=1)           # Shape: (B, T-1, vocab_size)

    # ── 5. generate ───────────────────────────────────────────────────────────
    @torch.no_grad()
    def generate(
        self,
        images: torch.Tensor,
        max_len: int = 35,
        bos_idx: int = 1,
        eos_idx: int = 2,
    ) -> torch.Tensor:
        """Greedy decoding."""
        features = self.encoder(images)
        h, c = self._init_hidden(features)
        B = images.size(0)

        # Start every sequence in the batch with BOS to signal the decoder
        # that generation should begin
        word = images.new_full((B,), bos_idx, dtype=torch.long)  # Shape: (B,)

        generated = []
        for _ in range(max_len):
            emb = self.embed(word)                  # Shape: (B, embed_dim)
            h, c = self.lstm(emb, (h, c))

            # Greedy decoding: always pick the single most likely next word.
            # This is fast but not optimal. Beam search would explore multiple
            # candidates and often produces better captions.
            word = self.fc(h).argmax(dim=-1)        # Shape: (B,)
            generated.append(word.unsqueeze(1))

        # Concatenate all predicted tokens into one sequence per batch item.
        # EOS truncation happens outside this function in eval_bleu and show_generated_captions.
        return torch.cat(generated, dim=1)          # Shape: (B, max_len)

## Training and Fine-tuning

1. **Model** → Instantiate ShowAndTellCaptioner and move it to the GPU
2. **Optimizer** → Configure Adam to update only the trainable parameters
3. **Scheduler** → Reduce the learning rate automatically when progress stalls
4. **Sanity Check** → Verify that freezing worked by counting trainable parameters
5. **Training** → Run the initial training loop with frozen encoder
6. **Fine-tuning** → Unfreeze the encoder and continue training with a smaller learning rate
7. **Fine-tuning Sanity Check** → Verify that the encoder is now trainable

---
*Core idea: First train only the decoder until it is stable, then unfreeze the encoder and fine-tune the whole model at a much smaller learning rate to avoid destroying the pretrained features. Early stopping prevents wasting compute on plateaued training*


In [ ]:
# ── 1. Model ──────────────────────────────────────────────────────────────────
# .to(DEVICE) moves all model weights to the GPU so that forward and backward
# passes do not require expensive data transfers between CPU and GPU every step
sat_model = ShowAndTellCaptioner(
    vocab_size=VOCAB_SIZE,
    encoder_dim=SAT_ENCODER_DIM,
    embed_dim=SAT_EMBED_DIM,
    decoder_dim=SAT_DECODER_DIM,
    dropout=SAT_DROPOUT,
    freeze_encoder=SAT_FREEZE,
).to(DEVICE)

# ── 2. Optimizer ──────────────────────────────────────────────────────────────
# filter(lambda p: p.requires_grad, ...) passes only the unfrozen parameters
# to Adam. Without this, Adam would allocate momentum buffers for the frozen
# ResNet weights too, wasting GPU memory without any benefit.
# weight_decay adds L2 regularization which penalizes large weights and
# reduces overfitting by keeping the decoder weights small.
sat_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, sat_model.parameters()),
    lr=SAT_LR, weight_decay=SAT_WD,
)

# ── 3. Scheduler ──────────────────────────────────────────────────────────────
# ReduceLROnPlateau watches the validation perplexity and multiplies the
# learning rate by factor=0.5 whenever it has not improved for patience=3 epochs.
# This lets the model take large steps early in training and smaller,
# more careful steps once it starts to converge.
sat_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    sat_optimizer, patience=3, factor=0.5
)

# ── 4. Sanity Check ───────────────────────────────────────────────────────────
# Counting trainable parameters confirms that the ResNet backbone is actually
# frozen. If this number is unexpectedly large, freeze_encoder likely did not work.
# p.numel() returns the total number of scalar values in a parameter tensor.
print(f'Trainable parameters: {sum(p.numel() for p in sat_model.parameters() if p.requires_grad):,}')

# ── 5. Training ───────────────────────────────────────────────────────────────
sat_losses, sat_ppls = train_model(
    sat_model, train_loader, test_loader,
    sat_optimizer, sat_scheduler, DEVICE,
    num_epochs=SAT_EPOCHS,
    model_name='show_and_tell',
    early_stopping_patience=5,
    config={"model": "show_and_tell", "lr": SAT_LR, "weight_decay": SAT_WD,
            "embed_dim": SAT_EMBED_DIM, "decoder_dim": SAT_DECODER_DIM,
            "dropout": SAT_DROPOUT, "epochs": SAT_EPOCHS},
)

plot_curves(sat_losses, sat_ppls, 'Show and Tell')

# ── 6. Fine-tuning ────────────────────────────────────────────────────────────
# Now that the decoder is stable we unfreeze the encoder so the ResNet backbone
# can adapt its features to the captioning task. The learning rate is 30x smaller
# than during initial training to avoid overwriting the pretrained features too aggressively.
for p in sat_model.encoder.body.parameters():
    p.requires_grad_(True)

sat_optimizer_ft = optim.Adam(
    filter(lambda p: p.requires_grad, sat_model.parameters()),
    lr=SAT_FINETUNE_LR,
    weight_decay=SAT_WD,
)
sat_scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(
    sat_optimizer_ft, patience=3, factor=0.5
)

# ── 7. Fine-tuning Sanity Check ───────────────────────────────────────────────
# The parameter count should now include the full ResNet body in addition to
# the decoder. If it did not increase, the unfreeze above did not work.
print(f'Trainable parameters after unfreeze: {sum(p.numel() for p in sat_model.parameters() if p.requires_grad):,}')

sat_losses_ft, sat_ppls_ft = train_model(
    sat_model, train_loader, test_loader,
    sat_optimizer_ft, sat_scheduler_ft, DEVICE,
    num_epochs=SAT_FINETUNE_EPOCHS, model_name='show_and_tell_finetuned',
    config={"model": "show_and_tell_finetuned", "lr": SAT_FINETUNE_LR,
            "weight_decay": SAT_WD, "phase": "finetune", "epochs": SAT_FINETUNE_EPOCHS},
)

plot_curves(sat_losses_ft, sat_ppls_ft, 'Show and Tell Fine-tuned')

## Evaluation of Best Checkpoint

1. **Load Checkpoint** → Restore the weights that achieved the lowest validation perplexity
2. **Quantitative Evaluation** → Compute perplexity and BLEU scores on the test set
3. **Qualitative Evaluation** → Visually inspect generated captions next to images

---
*Core idea: Always evaluate the best saved checkpoint rather than the final epoch because the last epoch is not necessarily the best one.*

In [ ]:
# ── 1. Load Checkpoint ────────────────────────────────────────────────────────
# Loads the weights saved by train_model whenever validation perplexity improved.
# map_location=DEVICE ensures the checkpoint loads correctly regardless of whether
# it was saved on a GPU and is now being loaded on a CPU or a different GPU.
sat_model.load_state_dict(torch.load('best_show_and_tell.pth', map_location=DEVICE))

# ── 2. Quantitative Evaluation ────────────────────────────────────────────────
# Perplexity measures how uncertain the model is on average at each word.
# BLEU measures how much the generated captions overlap with the human references.
# Both are needed because a low perplexity does not guarantee readable captions.
sat_train_ppl = eval_perplexity(sat_model, train_loader, DEVICE)
sat_ppl       = eval_perplexity(sat_model, test_loader,  DEVICE)
sat_bleu      = eval_bleu(sat_model, test_loader, tokenizer, DEVICE)

print(f'Train Perplexity: {sat_train_ppl:.2f}')
print(f'Test Perplexity:  {sat_ppl:.2f}')
for k, v in sat_bleu.items():
    print(f'  {k}: {v:.4f}')

# ── 3. Qualitative Evaluation ─────────────────────────────────────────────────
# Numbers alone can be misleading. Looking at actual generated captions reveals
# failure modes that perplexity and BLEU cannot capture, such as grammatically
# correct but semantically wrong descriptions.
show_generated_captions(sat_model, test_loader, tokenizer, DEVICE,
                        n=6, title='Show and Tell Generated Captions')

Training ran for 35 of the planned 40 epochs, with early stopping triggered after no improvement for 5 consecutive epochs. Validation-proxy perplexity dropped from 36.68 to 15.14, with the best checkpoint saved at epoch 30. Fine-tuning the encoder for another 10 epochs briefly improved validation-proxy perplexity to 14.99 at epoch 3, but it fluctuated above that value afterwards while training loss continued to fall. This indicates overfitting after unfreezing the encoder.

**Final metrics:**

| Metric    | Show & Tell |
| --------- | ----------- |
| Train PPL | 9.94        |
| Test PPL  | 14.71       |
| BLEU-1    | 0.5952      |
| BLEU-2    | 0.4156      |
| BLEU-3    | 0.2782      |
| BLEU-4    | 0.1863      |
| METEOR    | 0.3877      |

The gap between Train PPL (9.94) and Test PPL (14.71) suggests overfitting: the model assigns higher probability to training captions than to unseen test captions under teacher forcing.

The successful captions tended to describe familiar scene-level content, such as people on hills or dogs in outdoor scenes. The main errors were wrong colours, wrong activities, and generic fallback captions when the scene was visually uncommon or underrepresented in the training data.


## Model 2: Show, Attend and Tell

Attention-based captioning model from [Xu et al., 2015](https://arxiv.org/abs/1502.03044).  
- **Encoder**: pretrained ResNet-18, *spatial* feature maps (512 × 7 × 7 = 49 locations).  
- **Attention**: additive (Bahdanau) attention over spatial positions.  
- **Decoder**: single-layer LSTM conditioned on attended context at each step.

In [ ]:
# --- Hyperparameters ---
SATT_ENCODER_DIM   = 512
SATT_EMBED_DIM     = 256
SATT_DECODER_DIM   = 512

# The attention dimension is an intermediate projection size used to compute
# how much the decoder should focus on each spatial region of the image.
# It does not need to match encoder or decoder dim and is often kept smaller.
SATT_ATTENTION_DIM = 256

# 0.5 is the standard starting point for caption models because the decoder
# tends to overfit quickly on the limited vocabulary of image descriptions
SATT_DROPOUT       = 0.5

# Freezing the encoder protects the pretrained ResNet weights until the
# attention mechanism and decoder have learned something meaningful
SATT_FREEZE        = True

# 3e-4 is the commonly recommended default for Adam, known as
# "Karpathy's constant" in the community
SATT_LR            = 3e-4

SATT_EPOCHS        = 40
SATT_WD            = 1e-4

# 30x smaller than SATT_LR to avoid catastrophic forgetting: the backbone
# weights are already well-trained and only need minor adjustments
SATT_FINETUNE_LR     = 1e-5

# Fewer epochs than pretraining because the decoder and attention module
# are already stable and the encoder only needs small corrections
SATT_FINETUNE_EPOCHS = 10

# LR is halved after this many epochs without improvement in validation perplexity
SATT_SCHEDULER_PATIENCE = 3

# Multiplicative factor applied to the LR on each scheduler step
SATT_SCHEDULER_FACTOR   = 0.5

## Spatial CNN Encoder with Additive Attention and LSTM Decoder

1. **ShowAttendTellEncoder** → Extract spatial feature maps, one vector per image region
2. **AdditiveAttention** → Compute how much the decoder should focus on each region
3. **ShowAttendTellCaptioner** → Wire up encoder, attention and decoder
4. **_init_hidden** → Initialize LSTM state from the average of all spatial features
5. **forward** → Teacher-forced pass where each word attends to different image regions
6. **generate** → Greedy decoding returning both captions and attention maps

---
*Core idea: Instead of compressing the image into one vector, keep all spatial regions and let the decoder learn to focus on the relevant region at each word generation step.*

In [ ]:
# ── 1. ShowAttendTellEncoder ──────────────────────────────────────────────────
class ShowAttendTellEncoder(nn.Module):
    """ResNet-18 backbone preserving spatial feature maps."""

    def __init__(self, encoder_dim: int = 512, freeze: bool = True):
        super().__init__()

        # Transfer Learning: same pretrained ResNet-18 as Show and Tell but
        # this time we keep the spatial feature maps instead of pooling them
        # into a single vector, because attention needs one feature per region
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.body = nn.Sequential(*list(resnet.children())[:-2])
        self.encoder_dim = encoder_dim

        if freeze:
            for p in self.body.parameters():
                p.requires_grad_(False)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.body(images)              # Shape: (B, 512, H, W)
        B, C, H, W = features.shape

        # permute reorders from (B, C, H, W) to (B, H, W, C) so that the
        # spatial dimensions come first. reshape then flattens H and W into
        # one dimension so each of the H*W positions becomes one feature vector.
        # The result is a sequence of num_pixels vectors the attention can query.
        return features.permute(0, 2, 3, 1).reshape(B, H*W, C)  # Shape: (B, num_pixels, 512)


# ── 2. AdditiveAttention ──────────────────────────────────────────────────────
class AdditiveAttention(nn.Module):
    """Bahdanau (additive) attention over spatial image features."""

    def __init__(self, encoder_dim: int, decoder_dim: int, attention_dim: int):
        super().__init__()

        # Both the encoder output and the decoder hidden state are projected
        # into the same attention_dim space so they can be compared directly.
        # bias=False because the tanh nonlinearity below makes a bias redundant.
        self.W_enc = nn.Linear(encoder_dim,   attention_dim, bias=False)
        self.W_dec = nn.Linear(decoder_dim,   attention_dim, bias=False)

        # v collapses the attention_dim vector into a single scalar score per region
        self.v     = nn.Linear(attention_dim, 1,             bias=False)

    def forward(
        self,
        encoder_out:    torch.Tensor,   # Shape: (B, num_pixels, encoder_dim)
        decoder_hidden: torch.Tensor,   # Shape: (B, decoder_dim)
    ):
        # Project each spatial region independently into attention space
        enc_proj = self.W_enc(encoder_out)                          # Shape: (B, P, att_dim)

        # unsqueeze(1) adds a spatial dimension so the decoder state broadcasts
        # across all P regions when added to enc_proj
        dec_proj = self.W_dec(decoder_hidden).unsqueeze(1)          # Shape: (B, 1, att_dim)

        # tanh introduces nonlinearity so the model can learn complex relevance
        # relationships between the current decoder state and each image region.
        # This is the "additive" part that distinguishes Bahdanau from dot-product attention.
        energy  = self.v(torch.tanh(enc_proj + dec_proj)).squeeze(2) # Shape: (B, P)

        # Softmax turns the raw scores into a probability distribution over regions.
        # The weights sum to 1 across all spatial positions for each image in the batch.
        weights = torch.softmax(energy, dim=1)                       # Shape: (B, P)

        # The context vector is the weighted average of all spatial feature vectors.
        # Regions with higher attention weights contribute more to what the decoder sees.
        context = (encoder_out * weights.unsqueeze(2)).sum(1)        # Shape: (B, encoder_dim)
        return context, weights


# ── 3. ShowAttendTellCaptioner ────────────────────────────────────────────────
class ShowAttendTellCaptioner(nn.Module):
    """Show, Attend and Tell with additive attention."""

    def __init__(
        self,
        vocab_size:    int,
        encoder_dim:   int = 512,
        embed_dim:     int = 256,
        decoder_dim:   int = 512,
        attention_dim: int = 256,
        dropout:       float = 0.5,
        freeze_encoder: bool = True,
    ):
        super().__init__()
        self.encoder   = ShowAttendTellEncoder(encoder_dim, freeze_encoder)
        self.attention = AdditiveAttention(encoder_dim, decoder_dim, attention_dim)
        self.embed     = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.init_h    = nn.Linear(encoder_dim, decoder_dim)
        self.init_c    = nn.Linear(encoder_dim, decoder_dim)

        # The LSTM receives both the word embedding and the attended context vector
        # as input at every step. This is why the input size is embed_dim + encoder_dim
        # rather than just embed_dim as in Show and Tell.
        self.lstm      = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(decoder_dim, vocab_size)

    # ── 4. _init_hidden ───────────────────────────────────────────────────────
    def _init_hidden(self, encoder_out):
        # Average all spatial feature vectors into one global image representation.
        # This is different from Show and Tell where a single pooled vector was used.
        # Here we average over the num_pixels dimension to get the same information
        # without discarding the spatial structure needed for attention.
        mean_enc = encoder_out.mean(dim=1)                     # Shape: (B, encoder_dim)
        return torch.tanh(self.init_h(mean_enc)), torch.tanh(self.init_c(mean_enc))

    # ── 5. forward ────────────────────────────────────────────────────────────
    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        encoder_out = self.encoder(images)                     # Shape: (B, P, enc_dim)
        h, c = self._init_hidden(encoder_out)
        embeddings = self.embed(captions[:, :-1])              # Shape: (B, T-1, embed_dim)

        logits = []
        for t in range(embeddings.size(1)):
            # At every time step the attention mechanism looks at the current
            # decoder state h and decides which image regions are most relevant
            # for predicting the next word
            context, _ = self.attention(encoder_out, h)        # Shape: (B, encoder_dim)

            # Concatenate the word embedding with the context vector so the LSTM
            # receives both linguistic and visual information at every step
            lstm_in = torch.cat([self.dropout(embeddings[:, t]), context], dim=1)
            h, c    = self.lstm(lstm_in, (h, c))
            logits.append(self.fc(self.dropout(h)))

        return torch.stack(logits, dim=1)                      # Shape: (B, T-1, vocab_size)

    # ── 6. generate ───────────────────────────────────────────────────────────
    @torch.no_grad()
    def generate(
        self,
        images:  torch.Tensor,
        max_len: int = 35,
        bos_idx: int = 1,
        eos_idx: int = 2,
    ):
        """Greedy decoding. Returns (tokens, attention_maps)."""
        encoder_out = self.encoder(images)
        h, c = self._init_hidden(encoder_out)
        B    = images.size(0)
        word = images.new_full((B,), bos_idx, dtype=torch.long)

        generated, att_maps = [], []
        for _ in range(max_len):
            context, weights = self.attention(encoder_out, h)
            lstm_in = torch.cat([self.embed(word), context], dim=1)
            h, c    = self.lstm(lstm_in, (h, c))
            word    = self.fc(h).argmax(dim=-1)                # Shape: (B,)
            generated.append(word.unsqueeze(1))

            # Store the attention weights at every step so we can later
            # visualize which regions the model focused on for each generated word
            att_maps.append(weights.unsqueeze(1))

        # att_maps shape (B, T, P) lets you draw a heatmap per generated word
        return torch.cat(generated, dim=1), torch.cat(att_maps, dim=1)

## Training and Fine-tuning

1. **Model**                    → Instantiate ShowAttendTellCaptioner and move it to the GPU
2. **Optimizer**                → Configure Adam to update only the trainable parameters
3. **Scheduler**                → Reduce the learning rate automatically when progress stalls
4. **Sanity Check**             → Verify that freezing worked by counting trainable parameters
5. **Training**                 → Run the training loop with attention enabled and early stopping
6. **Fine-tuning**              → Unfreeze the encoder and continue training with a smaller LR
7. **Fine-tuning Sanity Check** → Verify that the encoder is now trainable

---
*Core idea: Same two-phase strategy as Show and Tell — freeze encoder first, then fine-tune end-to-end at a 30x smaller LR. Key difference: is_attention=True tells train_model that generate returns both tokens and attention weights and must be unpacked accordingly.*

In [ ]:
# ── 1. Model ──────────────────────────────────────────────────────────────────
# The only structural difference to Show and Tell is the additional
# attention_dim parameter which controls the size of the attention projection space.
satt_model = ShowAttendTellCaptioner(
    vocab_size=VOCAB_SIZE,
    encoder_dim=SATT_ENCODER_DIM,
    embed_dim=SATT_EMBED_DIM,
    decoder_dim=SATT_DECODER_DIM,
    attention_dim=SATT_ATTENTION_DIM,
    dropout=SATT_DROPOUT,
    freeze_encoder=SATT_FREEZE,
).to(DEVICE)

# ── 2. Optimizer ──────────────────────────────────────────────────────────────
# filter(lambda p: p.requires_grad, ...) excludes the frozen ResNet weights
# so Adam does not waste memory on momentum buffers for parameters that never update.
satt_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, satt_model.parameters()),
    lr=SATT_LR, weight_decay=SATT_WD,
)

# ── 3. Scheduler ──────────────────────────────────────────────────────────────
satt_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    satt_optimizer, patience=SATT_SCHEDULER_PATIENCE, factor=SATT_SCHEDULER_FACTOR
)

# ── 4. Sanity Check ───────────────────────────────────────────────────────────
# The parameter count should be slightly larger than Show and Tell because
# the attention module adds three additional linear layers W_enc, W_dec and v.
print(f'Trainable parameters: {sum(p.numel() for p in satt_model.parameters() if p.requires_grad):,}')

# ── 5. Training ───────────────────────────────────────────────────────────────
# is_attention=True tells train_model and eval_bleu that generate returns a tuple
# of (tokens, attention_weights) instead of just tokens and must be unpacked accordingly.
# early_stopping_patience=5 halts training if val_ppl does not improve for
# 5 consecutive epochs to avoid wasting compute on a plateaued model.
satt_losses, satt_ppls = train_model(
    satt_model, train_loader, test_loader,
    satt_optimizer, satt_scheduler, DEVICE,
    num_epochs=SATT_EPOCHS, model_name='show_attend_tell',
    is_attention=True,
    early_stopping_patience=5,
    config={"model": "show_attend_tell", "lr": SATT_LR, "weight_decay": SATT_WD,
        "embed_dim": SATT_EMBED_DIM, "decoder_dim": SATT_DECODER_DIM,
        "attention_dim": SATT_ATTENTION_DIM, "dropout": SATT_DROPOUT,
        "epochs": SATT_EPOCHS,
        "scheduler_patience": SATT_SCHEDULER_PATIENCE,
        "scheduler_factor": SATT_SCHEDULER_FACTOR},
)

plot_curves(satt_losses, satt_ppls, 'Show Attend and Tell')

# ── 6. Fine-tuning ────────────────────────────────────────────────────────────
# Now that the decoder and attention module are stable we unfreeze the encoder
# so the ResNet backbone can adapt its spatial features to the attention mechanism.
# The learning rate is 30x smaller than during initial training to avoid
# overwriting the pretrained features too aggressively (catastrophic forgetting).
for p in satt_model.encoder.body.parameters():
    p.requires_grad_(True)

satt_optimizer_ft = optim.Adam(
    filter(lambda p: p.requires_grad, satt_model.parameters()),
    lr=SATT_FINETUNE_LR,
    weight_decay=SATT_WD,
)
satt_scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(
    satt_optimizer_ft, patience=SATT_SCHEDULER_PATIENCE, factor=SATT_SCHEDULER_FACTOR
)

# ── 7. Fine-tuning Sanity Check ───────────────────────────────────────────────
# The parameter count should now include the full ResNet body in addition to
# the decoder and attention module. If it did not increase, the unfreeze above did not work.
print(f'Trainable parameters after unfreeze: {sum(p.numel() for p in satt_model.parameters() if p.requires_grad):,}')

satt_losses_ft, satt_ppls_ft = train_model(
    satt_model, train_loader, test_loader,
    satt_optimizer_ft, satt_scheduler_ft, DEVICE,
    num_epochs=SATT_FINETUNE_EPOCHS, model_name='show_attend_tell_finetuned',
    is_attention=True,
    early_stopping_patience=5,
    config={"model": "show_attend_tell_finetuned", "lr": SATT_FINETUNE_LR,
        "weight_decay": SATT_WD, "phase": "finetune",
        "epochs": SATT_FINETUNE_EPOCHS, 
        "scheduler_patience": SATT_SCHEDULER_PATIENCE,
        "scheduler_factor": SATT_SCHEDULER_FACTOR},
)

plot_curves(satt_losses_ft, satt_ppls_ft, 'Show Attend and Tell Fine-tuned')

> **Note on validation-proxy curves:** The right plot shows validation perplexity
> (PPL), which is mathematically equivalent to the validation loss via
> `Val Loss = ln(Val PPL)`. A falling PPL curve is a falling loss curve
> on a logarithmic scale, both carry identical information.

## Evaluation and Comparison

1. **Load Checkpoint** → Restore the weights that achieved the lowest validation-proxy perplexity
2. **Quantitative Evaluation** → Compute perplexity and BLEU scores for Show Attend and Tell
3. **Side-by-side Comparison** → Compare both models on all metrics in one table
4. **Qualitative Evaluation** → Visually inspect generated captions next to images

---
*Core idea: Evaluate both models under identical conditions so the comparison is fair, then check whether attention actually improves the quantitative metrics or only the interpretability.*

In [ ]:
# ── 1. Load Checkpoint ────────────────────────────────────────────────────────
# map_location=DEVICE ensures the checkpoint loads correctly regardless of
# whether it was saved on a different GPU or on CPU
satt_model.load_state_dict(torch.load('best_show_attend_tell.pth', map_location=DEVICE))

# ── 2. Quantitative Evaluation ────────────────────────────────────────────────
# is_attention=True is required because eval_bleu needs to unpack the tuple
# (tokens, attention_weights) that generate returns for attention models
satt_train_ppl = eval_perplexity(satt_model, train_loader, DEVICE)
satt_ppl       = eval_perplexity(satt_model, test_loader,  DEVICE)
satt_bleu      = eval_bleu(satt_model, test_loader, tokenizer, DEVICE, is_attention=True)

print(f'Train Perplexity: {satt_train_ppl:.2f}')
print(f'Test Perplexity:  {satt_ppl:.2f}')
for k, v in satt_bleu.items():
    print(f'  {k}: {v:.4f}')

# ── 3. Side-by-side Comparison ────────────────────────────────────────────────
# Attention does not always improve BLEU scores because BLEU measures n-gram
# overlap rather than visual grounding. The main advantage of attention is
# interpretability: you can see which image regions the model focused on
# for each generated word.
print('\n--- Comparison ---')
print(f'{"Metric":<12} {"Show&Tell":>12} {"ShowAttTell":>12}')
print(f'{"Train PPL":<12} {sat_train_ppl:>12.2f} {satt_train_ppl:>12.2f}')
print(f'{"Test PPL":<12} {sat_ppl:>12.2f} {satt_ppl:>12.2f}')
for k in sat_bleu:
    print(f'{k:<12} {sat_bleu[k]:>12.4f} {satt_bleu[k]:>12.4f}')

# ── 4. Qualitative Evaluation ─────────────────────────────────────────────────
# Comparing the generated captions visually with Show and Tell captions from
# Section 5 of Model 1 reveals whether attention produces more specific and
# accurate descriptions or just different formulations of the same generic captions
show_generated_captions(satt_model, test_loader, tokenizer, DEVICE,
                        n=6, is_attention=True, title='Show Attend and Tell Generated Captions')

Pretraining converged faster than Model 1, reaching its best checkpoint at
**epoch 13 (val-proxy PPL 16.49)** and triggering early stopping at epoch 18.
Fine-tuning improved val-proxy PPL to **15.90 at epoch 4**, then overfit, following
the same pattern as Model 1.

**Final metrics:**

| Metric    | Show & Tell | Show Attend & Tell |
|-----------|-------------|---------------------|
| Train PPL | 9.94        | 12.92               |
| Test PPL  | 14.71       | 16.49               |
| BLEU-1    | 0.5952      | 0.6206              |
| BLEU-2    | 0.4156      | 0.4327              |
| BLEU-3    | 0.2782      | 0.2903              |
| BLEU-4    | 0.1863      | 0.1915              |
| METEOR    | 0.3877      | 0.3767              |

Attention improved all BLEU scores, with gains ranging from +0.0052 on BLEU-4 to +0.0254 on BLEU-1. 
Test PPL worsened by 1.78. The Train–Test gap is smaller for SAT (1.28×) than for S&T (1.48×),
suggesting attention acts as a mild regulariser: the model does not fit training
data as tightly.

Attention helped on scene-level descriptions but did not fix the
skiing/climbing confusion or colour hallucinations. One new failure: repetitive
output ("a man is standing on a lake near a lake"), where attention revisits
the same region instead of moving on.

## Additional Task 1: Beam Search

**Hypothesis:** Beam search explores a wider set of token sequences than greedy decoding and should yield higher BLEU scores, especially at higher n-gram levels (BLEU-3/4).

## Beam Search Evaluation

1. **beam_search_single** → Find the best caption for one image by exploring multiple token sequences in parallel
2. **eval_bleu_beam** → Run beam search over the full dataset and compute BLEU and METEOR scores

---
*Core idea: Instead of always picking the single most likely next word, beam search keeps the best beam_width candidates at every step and selects the globally best sequence at the end.*

In [ ]:
def beam_search_single(
    model,
    image: torch.Tensor,      # (1, 3, H, W)
    tokenizer,
    beam_width: int = 5,
    max_len:    int = 35,
    is_attention: bool = False,
) -> list:
    """Beam search for a single image. Returns best token sequence."""

    # ── 1. Encode image ───────────────────────────────────────────────────────
    # The two branches initialise the decoder state differently because
    # attention decoders condition on the full spatial feature map while
    # the plain LSTM only needs a single pooled vector.
    device = image.device
    bos    = tokenizer.bos_idx
    eos    = tokenizer.eos_idx

    if is_attention:
        encoder_out = model.encoder(image)           # (1, P, enc_dim)
        h, c = model._init_hidden(encoder_out)
    else:
        features = model.encoder(image)              # (1, enc_dim)
        h = torch.tanh(model.init_h(features))
        c = torch.tanh(model.init_c(features))

    # ── 2. Initialise beams ───────────────────────────────────────────────────
    # Each beam carries its own hidden state so beams never share
    # mutable LSTM memory across different token histories.
    beams = [(0.0, [bos], h, c)]
    completed = []

    # ── 3. Expand candidates step by step ────────────────────────────────────
    for step in range(max_len):
        candidates = []
        for score, tokens, h_b, c_b in beams:
            if tokens[-1] == eos:
                completed.append((score, tokens))
                continue
            # new_tensor inherits device and dtype from image, avoiding an
            # explicit .to(device) call that would silently break on multi-GPU setups.
            word = image.new_tensor([tokens[-1]], dtype=torch.long)  # (1,)
            emb  = model.embed(word)                                  # (1, embed_dim)
            if is_attention:
                context, _ = model.attention(encoder_out, h_b)
                lstm_in = torch.cat([emb, context], dim=1)
            else:
                lstm_in = emb
            h_new, c_new = model.lstm(lstm_in, (h_b, c_b))
            # Log-probabilities are summed rather than raw probabilities
            # multiplied to prevent numerical underflow over long sequences.
            logprobs = torch.log_softmax(model.fc(h_new), dim=-1)    # (1, V)
            topk_lp, topk_idx = logprobs[0].topk(beam_width)
            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                candidates.append((score + lp, tokens + [idx], h_new, c_new))
        if not candidates:
            break
        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]

    # ── 4. Select best hypothesis ─────────────────────────────────────────────
    # Length normalisation prevents the search from favouring shorter sequences
    # because shorter sequences accumulate less negative log-probability
    # even if they are less fluent than longer ones
    completed += [(s, t) for s, t, *_ in beams]
    best = max(completed, key=lambda x: x[0] / max(len(x[1]), 1))
    toks = best[1][1:]   # skip BOS because it is a control token, not a real word
    if eos in toks:
        toks = toks[:toks.index(eos)]
    return toks


@torch.no_grad()
def eval_bleu_beam(
    model,
    loader,
    tokenizer,
    device,
    beam_width: int = 5,
    max_len:    int = 35,
    is_attention: bool = False,
):
    # ── 1. Collect hypotheses ─────────────────────────────────────────────────
    # imgs[i:i+1] preserves the batch dimension so beam_search_single
    # receives a proper 4-D tensor without an extra unsqueeze inside the function.
    model.eval()
    references, hypotheses = [], []
    captions_map = loader.dataset.captions_map
    for imgs, _, _, img_ids, _ in loader:
        imgs = imgs.to(device)
        for i, img_id in enumerate(img_ids):
            toks = beam_search_single(
                model, imgs[i:i+1], tokenizer,
                beam_width=beam_width, max_len=max_len,
                is_attention=is_attention,
            )
            refs = [tokenize(c) for c in captions_map[img_id]]
            references.append(refs)
            hyp = tokenizer.decode(toks, skip_special_tokens=True).split()
            hypotheses.append(hyp)

    # ── 2. Compute BLEU and METEOR scores ─────────────────────────────────────
    # Smoothing is applied because short hypotheses would otherwise
    # receive a zero score for any n-gram order with no matching reference span.
    sf = SmoothingFunction().method1

    # METEOR is included alongside BLEU to capture synonym matches that
    # pure n-gram overlap would miss
    meteor = sum(
        meteor_score(refs, hyp)
        for refs, hyp in zip(references, hypotheses)
    ) / len(hypotheses)

    return {
        'BLEU-1': corpus_bleu(references, hypotheses, (1,0,0,0),        smoothing_function=sf),
        'BLEU-2': corpus_bleu(references, hypotheses, (.5,.5,0,0),      smoothing_function=sf),
        'BLEU-3': corpus_bleu(references, hypotheses, (1/3,1/3,1/3,0),  smoothing_function=sf),
        'BLEU-4': corpus_bleu(references, hypotheses, (.25,.25,.25,.25), smoothing_function=sf),
        'METEOR': meteor,
        }

## Beam Search vs Greedy Decoding

1. **Configure** → Set beam width hyperparameter
2. **Evaluate** → Run beam search on both models over the full test set
3. **Compare** → Print greedy and beam BLEU scores side by side for both models

---
*Core idea: Test whether beam search produces higher BLEU scores than greedy decoding by exploring multiple candidate sequences simultaneously instead of always committing to the single most likely next word.*

In [ ]:
# ── 1. Configure beam width ───────────────────────────────────────────────────
# Width 5 is a common default that balances sequence quality against the
# quadratic growth in candidates per step.
BEAM_WIDTH = 5

# ── 2. Run beam evaluation for both models ────────────────────────────────────
# is_attention separates the two encoder modes so a single eval function
# covers both architectures without duplicating the BLEU logic.
sat_bleu_beam  = eval_bleu_beam(sat_model,  test_loader, tokenizer, DEVICE,
                                 beam_width=BEAM_WIDTH, is_attention=False)
satt_bleu_beam = eval_bleu_beam(satt_model, test_loader, tokenizer, DEVICE,
                                 beam_width=BEAM_WIDTH, is_attention=True)

# ── 3. Print comparison table ─────────────────────────────────────────────────
# Fixed column widths align greedy and beam columns so the score delta
# is readable at a glance without manual inspection.
print(f'\n--- Beam Search (k={BEAM_WIDTH}) vs Greedy ---')
print(f'{"Metric":<12} {"S&T greedy":>12} {"S&T beam":>12} {"SAT greedy":>12} {"SAT beam":>12}')
for k in sat_bleu:
    print(f'{k:<12} {sat_bleu[k]:>12.4f} {sat_bleu_beam[k]:>12.4f} '
          f'{satt_bleu[k]:>12.4f} {satt_bleu_beam[k]:>12.4f}')

Beam search with (k=5) improved most metrics on both models, especially the higher-order BLEU scores and METEOR.

| Metric | S&T greedy | S&T beam | SAT greedy | SAT beam |
| ------ | ---------- | -------- | ---------- | -------- |
| BLEU-1 | 0.5952     | 0.6067   | 0.6206     | 0.6174   |
| BLEU-2 | 0.4156     | 0.4317   | 0.4327     | 0.4376   |
| BLEU-3 | 0.2782     | 0.3023   | 0.2903     | 0.3062   |
| BLEU-4 | 0.1863     | 0.2088   | 0.1915     | 0.2110   |
| METEOR | 0.3877     | 0.4173   | 0.3767     | 0.4020   |

BLEU-4 improved by +12.1% for S&T and +10.2% for SAT, which shows that beam search mainly helps at the phrase level by keeping multiple candidate sequences instead of committing to a locally optimal next token. The slight BLEU-1 drop for SAT is small and may reflect different lexical choices rather than a clear loss in caption quality.


## Additional Task 2: Attention Visualization

**Hypothesis:** The attention mechanism learns to focus on semantically relevant image regions for each generated word (e.g., attending to a dog when generating the word *dog*).

## Attention Visualization Function

1. **Generate** → Run the attention model on one image to get tokens and attention weights
2. **Truncate** → Cut sequence at EOS and align tokens with their attention maps
3. **Plot** → Show the original image followed by one heatmap overlay per generated word

---
*Core idea: For each generated word, reshape the attention weights back into a spatial grid and overlay them on the original image to reveal which region the model was looking at when it produced that word.*

In [ ]:
def visualize_attention(
    model,
    image_tensor: torch.Tensor,   # Shape: (1, 3, H, W) normalised
    raw_image:    Image.Image,     # original PIL image for display
    tokenizer,
    device,
    max_len: int = 20,
    spatial_size: int = 7,
):
    """Overlay attention heatmaps on the original image for each generated word."""

    # ── 1. Generate ───────────────────────────────────────────────────────────
    model.eval()
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        gen_tokens, att_maps = model.generate(
            image_tensor, max_len=max_len,
            bos_idx=tokenizer.bos_idx, eos_idx=tokenizer.eos_idx,
        )
        # gen_tokens: Shape (1, T)
        # att_maps:   Shape (1, T, P) where P = spatial_size * spatial_size

    # ── 2. Truncate ───────────────────────────────────────────────────────────
    toks = gen_tokens[0].cpu().tolist()
    atts = att_maps[0].cpu().numpy()           # Shape: (T, P)

    # Remove everything after EOS so we only visualize real generated words
    # and not the padding that follows the end of the sequence
    if tokenizer.eos_idx in toks:
        end = toks.index(tokenizer.eos_idx)
        toks, atts = toks[:end], atts[:end]

    words = tokenizer.decode(toks, skip_special_tokens=True).split()
    T = min(len(words), len(atts))

    # ── 3. Plot ───────────────────────────────────────────────────────────────
    # One extra panel at the start shows the original image without any overlay
    # so the viewer has a clean reference to compare the heatmaps against
    cols = 4
    rows = math.ceil(T / cols) + 1
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = axes.flatten()

    axes[0].imshow(raw_image)
    axes[0].set_title('Original', fontsize=9)
    axes[0].axis('off')

    img_w, img_h = raw_image.size
    for t in range(T):
        # Reshape the flat attention vector back into a 2D spatial grid
        att = atts[t].reshape(spatial_size, spatial_size)

        # Normalise to [0, 1] so the colormap uses the full range regardless
        # of how peaked or flat the attention distribution is.
        # The 1e-8 prevents division by zero when all weights are identical.
        att = (att - att.min()) / (att.max() - att.min() + 1e-8)

        # Upsample the 7x7 attention grid to the full image resolution using
        # bilinear interpolation so the overlay aligns with the original pixels
        att_up = np.array(
            Image.fromarray((att * 255).astype(np.uint8)).resize(
                (img_w, img_h), Image.BILINEAR
            )
        ) / 255.0

        # Overlay the heatmap on the original image with alpha=0.5 so both
        # the image content and the attention region remain visible
        axes[t + 1].imshow(raw_image)
        axes[t + 1].imshow(att_up, cmap='jet', alpha=0.5)
        axes[t + 1].set_title(words[t], fontsize=10, fontweight='bold')
        axes[t + 1].axis('off')

    # Hide unused subplot panels when the number of words is not a multiple of cols
    for ax in axes[T + 1:]:
        ax.axis('off')

    plt.suptitle('Attention Visualization Show Attend and Tell', fontsize=12)
    plt.tight_layout()
    plt.show()
    print('Generated:', ' '.join(words))

## Attention Visualization on Sample Images

1. **Sample** → Randomly pick 3 test images to visualize
2. **Load and Transform** → Open each image and prepare it for the model
3. **Visualize** → Overlay attention heatmaps for each generated word

---
*Core idea: Randomly sampling from the test set gives an unbiased look at where the model attends, rather than cherry-picking images where attention looks convincing.*

In [ ]:
# ── 1. Sample ─────────────────────────────────────────────────────────────────
from dataloader import load_split_image_ids
from pathlib import Path
import random

# Load all test image IDs and randomly pick 3 to visualize.
# Random sampling avoids the bias of manually selecting images where
# the attention maps look good, giving a more honest picture of model behavior.
test_ids = load_split_image_ids('test', data_dir)
sampled  = random.sample(test_ids, 3)

# ── 2. Load and Transform ─────────────────────────────────────────────────────
for img_id in sampled:
    # raw_img is kept as a PIL image for display because matplotlib expects
    # pixel values in [0, 1] or [0, 255], not the normalized tensor
    raw_img = Image.open(Path(data_dir) / 'images' / img_id)

    # build_test_transforms() applies the same deterministic preprocessing as
    # during evaluation. unsqueeze(0) adds the batch dimension the model expects.
    img_tensor = build_test_transforms()(raw_img).unsqueeze(0)  # Shape: (1, 3, 224, 224)

    # ── 3. Visualize ──────────────────────────────────────────────────────────
    visualize_attention(satt_model, img_tensor, raw_img, tokenizer, DEVICE)

The attention maps show that the model generally focuses on semantically
relevant regions for content words, partially confirming the hypothesis.

**Image 1** ("a dog runs through the water"): "a" and "dog" correctly attend
to the dog subjects. "runs" shifts focus to the wave area rather than staying
on the dogs (the model associates the action with its environmental context
instead of the acting subject). "water" correctly attends to the right side of
the frame. The caption is factually correct despite the misplaced attention for
"runs": correct output does not guarantee correct localisation. The second dog
is missed entirely.

**Image 2** ("a skateboarder is doing a trick on a ramp"): "ramp" correctly
attends to the ramp in the lower half. "skateboarder" however focuses on the
ramp area rather than the person silhouette (the model confuses the subject
with its associated object). Function words ("is", "a", "on") show diffuse,
unfocused attention as expected. The caption is factually correct despite
imperfect subject localisation.

**Image 3** ("a man in a red shirt is standing on a red"): "red" and "shirt"
correctly attend to the orange-red clothing. "man" focuses on the person but
misclassifies gender. The caption ends abruptly with a repeated "red" (the
model loops back to the dominant colour of the carnival background and hits
the length limit without generating EOS).

**Overall:** Attention helps with colour and object localisation but remains
coarse (7×7 grid) for fine-grained distinctions. BLEU rewards correct output,
not correct localisation: a caption can score well while the attention tells
a completely different story.

## Additional Task 3: Pretrained Word Embeddings (GloVe)

**Hypothesis:** Initialising the embedding layer with GloVe-100 vectors provides better semantic structure in the embedding space and should improve early-epoch convergence and final BLEU scores, particularly for less frequent words.

## GloVe Embedding Setup

1. **Download & extract** → Fetch GloVe zip and unpack the chosen dimension file
2. **Build embedding matrix** → Initialise vocab-sized matrix and overwrite with GloVe vectors
3. **Load matrix** → Call load_glove_matrix for the current tokenizer vocabulary

---
*Core idea: Words not found in GloVe keep a small random initialisation so the embedding layer can still learn representations for out-of-vocabulary tokens during training.*

In [ ]:
# ── 1. Download & extract ─────────────────────────────────────────────────────
# The file check avoids re-downloading on every kernel restart, which
# matters because the full zip is 860 MB.
GLOVE_DIM  = 100
GLOVE_SCHEDULER_PATIENCE = 3
GLOVE_SCHEDULER_FACTOR   = 0.5
GLOVE_URL  = 'https://nlp.stanford.edu/data/glove.6B.zip'
GLOVE_ZIP  = 'glove.6B.zip'
GLOVE_FILE = f'glove.6B.{GLOVE_DIM}d.txt'

if not os.path.exists(GLOVE_FILE):
    print('Downloading GloVe…')
    urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP)
    with zipfile.ZipFile(GLOVE_ZIP) as z:
        z.extract(GLOVE_FILE)
    print('Done.')


def load_glove_matrix(glove_file: str, stoi: dict, embed_dim: int) -> torch.Tensor:
    """Return embedding matrix (vocab_size, embed_dim) initialised from GloVe."""

    # ── 2. Build embedding matrix ─────────────────────────────────────────────
    # Normal init with small variance gives OOV tokens a learnable starting
    # point instead of a dead zero vector that gradient updates cannot escape.
    matrix = torch.zeros(len(stoi), embed_dim)
    nn.init.normal_(matrix, 0, 0.01)
    found = 0
    with open(glove_file, encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word  = parts[0]
            if word in stoi:
                matrix[stoi[word]] = torch.tensor([float(x) for x in parts[1:]])
                found += 1
    print(f'GloVe coverage: {found}/{len(stoi)} tokens ({100*found/len(stoi):.1f}%)')
    return matrix


# ── 3. Load matrix ────────────────────────────────────────────────────────────
# stoi maps each token string to its integer index, which is exactly the
# row order the embedding layer expects.
glove_matrix = load_glove_matrix(GLOVE_FILE, tokenizer.stoi, GLOVE_DIM)

## Show and Tell with GloVe Embeddings

1. **Build model** → Instantiate ShowAndTellCaptioner with GloVe-matching embed_dim
2. **Inject GloVe weights** → Copy pre-trained vectors into the embedding layer
3. **Configure optimiser** → Adam with frozen-parameter filter and LR scheduler
4. **Train** → Run training loop and plot learning curves

---
*Core idea: Copying GloVe vectors before training gives the embedding layer a semantically meaningful starting point, so the model can focus on learning the decoder rather than word representations from scratch.*

In [ ]:
# ── 1. Build model ────────────────────────────────────────────────────────────
# embed_dim must match GLOVE_DIM exactly so the pre-trained vectors fit
# the embedding layer without a projection step.
glove_model = ShowAndTellCaptioner(
    vocab_size=VOCAB_SIZE,
    encoder_dim=SAT_ENCODER_DIM,
    embed_dim=GLOVE_DIM,
    decoder_dim=SAT_DECODER_DIM,
    dropout=SAT_DROPOUT,
    freeze_encoder=SAT_FREEZE,
).to(DEVICE)

# ── 2. Inject GloVe weights ───────────────────────────────────────────────────
# no_grad prevents copy_ from being recorded in the autograd graph,
# which would incorrectly treat the initialisation as a learnable operation.
with torch.no_grad():
    glove_model.embed.weight.copy_(glove_matrix)

# ── 3. Configure optimiser ────────────────────────────────────────────────────
# filter(requires_grad) excludes the frozen encoder parameters so Adam
# does not allocate momentum buffers for weights that never update.
glove_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, glove_model.parameters()),
    lr=SAT_LR, weight_decay=SAT_WD,
)
glove_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    glove_optimizer, patience=GLOVE_SCHEDULER_PATIENCE, factor=GLOVE_SCHEDULER_FACTOR,
)

# ── 4. Train ──────────────────────────────────────────────────────────────────
# Saving under a distinct model_name keeps the GloVe checkpoint separate
# from the baseline so both can be compared after training.
glove_losses, glove_ppls = train_model(
    glove_model, train_loader, test_loader,
    glove_optimizer, glove_scheduler, DEVICE,
    num_epochs=SAT_EPOCHS, model_name='glove_model',
    config={"model": "glove_show_and_tell", "lr": SAT_LR, "weight_decay": SAT_WD,
            "embed_dim": GLOVE_DIM, "decoder_dim": SAT_DECODER_DIM,
            "dropout": SAT_DROPOUT, "epochs": SAT_EPOCHS,
            "glove_dim": GLOVE_DIM,
            "scheduler_patience": GLOVE_SCHEDULER_PATIENCE,
            "scheduler_factor": GLOVE_SCHEDULER_FACTOR},
)

plot_curves(glove_losses, glove_ppls, 'Show and Tell + GloVe-100')

## GloVe vs. Random Init Comparison

1. **Load & evaluate** → Restore best checkpoint and compute perplexity and BLEU
2. **Print comparison** → Align random-init and GloVe scores in a fixed-width table

---
*Core idea: Comparing against the random-init baseline isolates the effect of GloVe pre-training on both fluency (perplexity) and n-gram overlap (BLEU).*

In [ ]:
# ── 1. Load & evaluate ────────────────────────────────────────────────────────
# map_location=DEVICE ensures the checkpoint loads correctly regardless of
# whether it was saved on a different GPU or on CPU.
glove_model.load_state_dict(torch.load('best_glove_model.pth', map_location=DEVICE))
glove_ppl  = eval_perplexity(glove_model, test_loader, DEVICE)
glove_bleu = eval_bleu(glove_model, test_loader, tokenizer, DEVICE)

# ── 2. Print comparison ───────────────────────────────────────────────────────
# sat_ppl and sat_bleu come from the random-init baseline evaluated earlier
# so both models are scored under identical conditions.
print('\n--- GloVe vs Random Init (Show and Tell) ---')
print(f'{"Metric":<12} {"Random":>12} {"GloVe-100":>12}')
print(f'{"Perplexity":<12} {sat_ppl:>12.2f} {glove_ppl:>12.2f}')
for k in sat_bleu:
    print(f'{k:<12} {sat_bleu[k]:>12.4f} {glove_bleu[k]:>12.4f}')

GloVe initialisation did not improve convergence speed or final scores.
The hypothesis is not confirmed.

| Metric     | Random Init | GloVe-100  |
|------------|-------------|------------|
| Perplexity | 14.71       | 15.24      |
| BLEU-1     | 0.5952      | 0.5761     |
| BLEU-2     | 0.4156      | 0.4019     |
| BLEU-3     | 0.2782      | 0.2682     |
| BLEU-4     | 0.1863      | 0.1768     |
| METEOR     | 0.3877      | **0.3897** |

GloVe even started worse (val PPL epoch 1: 41.57 vs. 36.68 for random init)
and reached its best checkpoint later (epoch 33 vs. 30). The marginal METEOR
gain (+0.002) is the only metric favouring GloVe and is likely within noise.

Two confounds limit the interpretation: (1) the GloVe model uses embed_dim=100
to match the vector dimension, while random init used embed_dim=256, which
reduces model capacity independently of initialisation. (2) GloVe coverage was
99.4% (2634/2649 tokens), so the missing 15 tokens likely include the special tokens and possibly dataset-specific tokens not covered by GloVe. With a small
vocabulary of 2649 words, random embeddings can be learned from scratch within
30 epochs, leaving little room for GloVe to provide an advantage.

## Additional Task 4: Dot-Product Attention

**Hypothesis:** Scaled dot-product attention (as in transformers) is computationally cheaper than additive attention and may converge comparably or better due to its multiplicative interaction.

## Dot-Product Attention Module and Model Variant

1. **DotProductAttention** → Replace additive attention with scaled dot-product attention
2. **ShowDotAttendTellCaptioner** → Swap only the attention module, keeping everything else identical

---
*Core idea: By replacing only the attention module and inheriting everything else from ShowAttendTellCaptioner, any difference in performance can be attributed purely to the attention mechanism and not to other architectural changes.*

In [ ]:
# ── 1. DotProductAttention ────────────────────────────────────────────────────
class DotProductAttention(nn.Module):
    """Scaled dot-product attention over spatial image features."""

    def __init__(self, encoder_dim: int, decoder_dim: int):
        super().__init__()

        # W_q projects the decoder hidden state into the same space as the
        # encoder output so the dot product is dimensionally compatible
        # without needing a separate key matrix like in Bahdanau attention
        self.W_q   = nn.Linear(decoder_dim, encoder_dim, bias=False)

        # Scaling by 1/sqrt(encoder_dim) prevents dot products from growing
        # so large that softmax saturates and gradients vanish during training
        self.scale = encoder_dim ** -0.5

    def forward(
        self,
        encoder_out:    torch.Tensor,  # Shape: (B, P, encoder_dim)
        decoder_hidden: torch.Tensor,  # Shape: (B, decoder_dim)
    ):
        # Project decoder state into encoder space and add a dimension
        # so it can be used as the right-hand matrix in the batch matrix multiply
        query  = self.W_q(decoder_hidden).unsqueeze(2)         # Shape: (B, encoder_dim, 1)

        # bmm computes the dot product between each spatial region and the query.
        # The result is one scalar score per region measuring how relevant it is.
        energy  = torch.bmm(encoder_out, query).squeeze(2)     # Shape: (B, P)

        # Multiply by scale before softmax to keep the variance of the scores
        # independent of encoder_dim, which stabilizes the attention distribution
        weights = torch.softmax(energy * self.scale, dim=1)    # Shape: (B, P)

        # Weighted average of all spatial feature vectors produces the context
        # vector that the decoder uses together with the word embedding
        context = (encoder_out * weights.unsqueeze(2)).sum(1)  # Shape: (B, encoder_dim)
        return context, weights


# ── 2. ShowDotAttendTellCaptioner ─────────────────────────────────────────────
class ShowDotAttendTellCaptioner(ShowAttendTellCaptioner):
    """Show, Attend and Tell with scaled dot-product attention."""

    def __init__(self, vocab_size, encoder_dim=512, embed_dim=256,
                 decoder_dim=512, attention_dim=256,
                 dropout=0.5, freeze_encoder=True):
        super().__init__(vocab_size, encoder_dim, embed_dim,
                         decoder_dim, attention_dim, dropout, freeze_encoder)

        # Replacing only self.attention keeps all other decoder components
        # identical so any performance difference between this model and
        # ShowAttendTellCaptioner is attributable solely to the attention mechanism
        # and not to any other architectural change
        self.attention = DotProductAttention(encoder_dim, decoder_dim)

## Dot-Product Attention Training

1. **Model** → Instantiate ShowDotAttendTellCaptioner with the same hyperparameters as Model 2
2. **Optimizer** → Configure Adam to update only the trainable parameters
3. **Scheduler** → Reduce the learning rate automatically when progress stalls
4. **Training** → Train with the same setup as Model 2 so the comparison is fair

---
*Core idea: Reuse all SATT hyperparameters so the only variable between this model and Model 2 is the attention mechanism, making the comparison scientifically valid.*

In [ ]:
# ── 1. Model ──────────────────────────────────────────────────────────────────
# Intentionally uses the same hyperparameters as ShowAttendTellCaptioner so
# that any difference in training behavior or BLEU scores can be attributed
# to the dot-product attention mechanism and nothing else
dot_model = ShowDotAttendTellCaptioner(
    vocab_size=VOCAB_SIZE,
    encoder_dim=SATT_ENCODER_DIM,
    embed_dim=SATT_EMBED_DIM,
    decoder_dim=SATT_DECODER_DIM,
    attention_dim=SATT_ATTENTION_DIM,
    dropout=SATT_DROPOUT,
    freeze_encoder=SATT_FREEZE,
).to(DEVICE)

# ── 2. Optimizer ──────────────────────────────────────────────────────────────
# attention_dim is accepted by the constructor but unused in DotProductAttention
# because dot-product attention has no projection into a separate attention space.
# It is kept for interface compatibility with the parent class.
dot_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, dot_model.parameters()),
    lr=SATT_LR, weight_decay=SATT_WD,
)

# ── 3. Scheduler ──────────────────────────────────────────────────────────────
# Reuses SATT scheduler constants to keep the comparison fair:
# any difference in results is due to the attention mechanism, not the scheduler
dot_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    dot_optimizer, patience=SATT_SCHEDULER_PATIENCE, factor=SATT_SCHEDULER_FACTOR
)

# ── 4. Training ───────────────────────────────────────────────────────────────
# is_attention=True is required because dot_model.generate also returns
# a tuple of (tokens, attention_weights) inherited from ShowAttendTellCaptioner
dot_losses, dot_ppls = train_model(
    dot_model, train_loader, test_loader,
    dot_optimizer, dot_scheduler, DEVICE,
    num_epochs=SATT_EPOCHS, model_name='dot_model',
    is_attention=True,
    config={"model": "dot_product_attention", "lr": SATT_LR, "weight_decay": SATT_WD,
            "embed_dim": SATT_EMBED_DIM, "decoder_dim": SATT_DECODER_DIM,
            "attention_type": "dot_product", "dropout": SATT_DROPOUT,
            "epochs": SATT_EPOCHS,
            "scheduler_patience": SATT_SCHEDULER_PATIENCE,
            "scheduler_factor": SATT_SCHEDULER_FACTOR},
)

plot_curves(dot_losses, dot_ppls, 'Dot-Product Attention')

## Attention Comparison

1. **Load Checkpoint** → Restore the best dot-product attention weights
2. **Evaluate** → Compute perplexity and BLEU scores for the dot-product model
3. **Compare** → Print additive vs dot-product attention side by side

---
*Core idea: Evaluating both attention models under identical conditions reveals whether the added complexity of additive attention justifies its higher parameter count compared to the simpler dot-product variant.*

In [ ]:
# ── 1. Load Checkpoint ────────────────────────────────────────────────────────
# map_location=DEVICE ensures the checkpoint loads correctly regardless of
# whether it was saved on a different GPU or on CPU
dot_model.load_state_dict(torch.load('best_dot_model.pth', map_location=DEVICE))

# ── 2. Evaluate ───────────────────────────────────────────────────────────────
# is_attention=True is required because eval_bleu must unpack the tuple
# (tokens, attention_weights) that the inherited generate method returns
dot_ppl  = eval_perplexity(dot_model, test_loader, DEVICE)
dot_bleu = eval_bleu(dot_model, test_loader, tokenizer, DEVICE, is_attention=True)

# ── 3. Compare ────────────────────────────────────────────────────────────────
# Additive attention has more parameters due to W_enc, W_dec and v.
# Dot-product attention only needs W_q, making it faster and leaner.
# If scores are similar, dot-product attention is the better choice
# because it achieves the same result with fewer parameters.
print('\n--- Additive vs Dot-Product Attention ---')
print(f'{"Metric":<12} {"Additive":>12} {"Dot-Product":>14}')
print(f'{"Perplexity":<12} {satt_ppl:>12.2f} {dot_ppl:>14.2f}')
for k in satt_bleu:
    print(f'{k:<12} {satt_bleu[k]:>12.4f} {dot_bleu[k]:>14.4f}')

Dot-product attention achieved better perplexity and METEOR than additive attention, but slightly lower BLEU-1 to BLEU-3 and almost identical BLEU-4. The hypothesis is therefore only partially confirmed.

| Metric     | Additive   | Dot-Product |
|------------|------------|-------------|
| Perplexity | 16.49      | **15.21**   |
| BLEU-1     | **0.6206** | 0.6018      |
| BLEU-2     | **0.4327** | 0.4229      |
| BLEU-3     | **0.2903** | 0.2861      |
| BLEU-4     | **0.1915** | 0.1902      |
| METEOR     | 0.3767     | **0.3961**  |

Dot-product attention kept improving throughout all 40 epochs (best: epoch 39,
PPL 15.21), whereas additive attention stopped at epoch 13. This suggests
dot-product converges more slowly but more steadily. The METEOR gain (+0.019)
indicates better synonym-level word selection, while the BLEU gap is
negligible (−0.001 on BLEU-4). For equal or better performance with a
simpler, parameter-lean mechanism (one linear layer vs. three).
Dot-product attention is architecturally simpler and improves perplexity and METEOR in this setup, but it is not clearly superior overall because BLEU-1 to BLEU-3 decrease slightly and BLEU-4 remains almost unchanged.

## Additional Task 5: Transformer-Based Decoder

**Hypothesis:** A transformer decoder with multi-head cross-attention over spatial image features can better model long-range token dependencies than an LSTM, leading to more grammatical and globally coherent captions.

## Transformer Captioner

1. **PositionalEncoding** → Inject position information into token embeddings since Transformers have no built-in sense of order
2. **TransformerCaptioner.__init__** → Wire up encoder, positional encoding, Transformer decoder and projection head
3. **_encode** → Extract spatial features and project them into the decoder dimension
4. **forward** → Teacher-forced pass through the Transformer decoder with causal masking
5. **generate** → Autoregressive greedy decoding one token at a time

---
*Core idea: Replace the LSTM decoder with a Transformer decoder that uses self-attention over previously generated tokens and cross-attention over all spatial image regions simultaneously, rather than processing the sequence step by step.*

In [ ]:
# ── 1. PositionalEncoding ─────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 200, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Transformers process all tokens in parallel and have no recurrence,
        # so they cannot distinguish "dog" at position 2 from "dog" at position 5.
        # Positional encoding adds a unique signal to each position so the model
        # can learn word order from the embeddings alone.
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()

        # The alternating sin/cos pattern creates a unique fingerprint per position.
        # div controls the frequency: early dimensions oscillate fast, later ones slow.
        # PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
        # PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        # register_buffer saves pe as part of the model state without making it
        # a trainable parameter. It moves to GPU automatically with .to(device).
        self.register_buffer('pe', pe.unsqueeze(0))  # Shape: (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Slice pe to the actual sequence length so the function works for
        # any input shorter than max_len without padding or truncation
        return self.dropout(x + self.pe[:, :x.size(1)])


# ── 2. TransformerCaptioner.__init__ ──────────────────────────────────────────
class TransformerCaptioner(nn.Module):
    """Transformer decoder with cross-attention to ResNet-18 spatial features."""

    def __init__(
        self,
        vocab_size:   int,
        encoder_dim:  int   = 512,
        d_model:      int   = 256,
        nhead:        int   = 4,
        num_layers:   int   = 2,
        dim_ff:       int   = 1024,
        dropout:      float = 0.1,
        freeze_encoder: bool = True,
        max_len:      int   = 60,
    ):
        super().__init__()

        # Reuse the same spatial encoder as ShowAttendTellCaptioner because
        # the Transformer decoder also needs one feature vector per image region
        self.encoder  = ShowAttendTellEncoder(encoder_dim, freeze_encoder)

        # The encoder produces 512-dimensional features but the Transformer
        # operates in d_model dimensions, so a linear projection bridges the gap
        self.enc_proj = nn.Linear(encoder_dim, d_model)

        self.embed    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc  = PositionalEncoding(d_model, max_len, dropout)

        # TransformerDecoderLayer contains three sub-layers:
        # (1) masked self-attention over previously generated tokens
        # (2) cross-attention over all spatial image regions
        # (3) a feedforward network applied position-wise
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True,
        )
        self.decoder  = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc       = nn.Linear(d_model, vocab_size)
        self.d_model  = d_model

    # ── 3. _encode ────────────────────────────────────────────────────────────
    def _encode(self, images: torch.Tensor):
        enc = self.encoder(images)             # Shape: (B, P, encoder_dim)
        return self.enc_proj(enc)              # Shape: (B, P, d_model)

    # ── 4. forward ────────────────────────────────────────────────────────────
    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        memory = self._encode(images)          # Shape: (B, P, d_model)

        # Drop the last token for teacher forcing: the decoder sees BOS through
        # the second-to-last token and predicts positions 1 through T
        tgt     = captions[:, :-1]             # Shape: (B, T-1)

        # Scale embeddings by sqrt(d_model) to keep their magnitude comparable
        # to the positional encoding, which has values in [-1, 1]
        tgt_emb = self.pos_enc(self.embed(tgt) * math.sqrt(self.d_model))

        T = tgt_emb.size(1)

        # The causal mask prevents each position from attending to future tokens.
        # Without it the model could cheat during training by looking ahead.
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=images.device)

        # tgt_key_padding_mask tells the attention layers to ignore PAD positions
        # so padding tokens do not influence the attention distribution
        tgt_pad_mask = (tgt == PAD_IDX)        # Shape: (B, T-1)

        out = self.decoder(tgt_emb, memory,
                           tgt_mask=causal_mask,
                           tgt_key_padding_mask=tgt_pad_mask)  # Shape: (B, T-1, d_model)
        return self.fc(out)                    # Shape: (B, T-1, vocab_size)

    # ── 5. generate ───────────────────────────────────────────────────────────
    @torch.no_grad()
    def generate(
        self,
        images:  torch.Tensor,
        max_len: int = 35,
        bos_idx: int = 1,
        eos_idx: int = 2,
    ) -> torch.Tensor:
        """Autoregressive greedy decoding."""
        memory = self._encode(images)          # Shape: (B, P, d_model)
        B      = images.size(0)

        # Start with BOS and grow the sequence by one token per step.
        # Unlike the LSTM which maintains a hidden state, the Transformer
        # re-reads the entire token sequence at every step.
        tokens = images.new_full((B, 1), bos_idx, dtype=torch.long)

        for _ in range(max_len):
            tgt_emb = self.pos_enc(
                self.embed(tokens) * math.sqrt(self.d_model)
            )                                  # Shape: (B, t, d_model)
            T = tgt_emb.size(1)
            causal_mask = nn.Transformer.generate_square_subsequent_mask(
                T, device=images.device
            )
            out      = self.decoder(tgt_emb, memory, tgt_mask=causal_mask)

            # Take only the last position because that is where the prediction
            # for the next token lives after attending to all previous tokens
            next_tok = self.fc(out[:, -1]).argmax(dim=-1, keepdim=True)  # Shape: (B, 1)
            tokens   = torch.cat([tokens, next_tok], dim=1)

            # Stop as soon as every sequence in the batch has produced EOS
            if (next_tok == eos_idx).all():
                break

        # Strip BOS from the output because it is a control token and not
        # part of the generated caption
        return tokens[:, 1:]                   # Shape: (B, generated_len)

## Transformer Training

1. **Hyperparameters** → Define Transformer-specific architecture and training constants
2. **Model** → Instantiate TransformerCaptioner and move it to the GPU
3. **Optimizer** → Configure Adam with a smaller learning rate than the LSTM models
4. **Scheduler** → Reduce the learning rate automatically when progress stalls
5. **Sanity Check** → Verify parameter count and that freezing worked
6. **Training** → Train the Transformer decoder with the same loop as the LSTM models

---
*Core idea: Use the same training infrastructure as Model 1 and 2 so any difference in results is attributable to the Transformer architecture and not to differences in training setup.*

In [ ]:
# ── 1. Hyperparameters ────────────────────────────────────────────────────────
TF_D_MODEL = 256   # embedding and decoder dimension throughout the Transformer
TF_NHEAD   = 4     # number of parallel attention heads, must divide d_model evenly
TF_SCHEDULER_PATIENCE = 3
TF_SCHEDULER_FACTOR   = 0.5

# Two decoder layers keep the model lightweight given the small dataset.
# More layers would increase capacity but also the risk of overfitting.
TF_LAYERS  = 2

# dim_ff is the hidden size of the feedforward sublayer inside each decoder layer.
# It is typically 4x d_model following the original Transformer paper.
TF_DIM_FF  = 1024

# Transformers are more sensitive to dropout than LSTMs because attention can
# distribute weight across many positions at once, making regularization critical.
TF_DROPOUT = 0.1

# A smaller learning rate than the LSTM models because Transformers are more
# sensitive to large gradient updates early in training
TF_LR      = 1e-4

TF_EPOCHS  = 20
TF_WD      = 1e-4

# ── 2. Model ──────────────────────────────────────────────────────────────────
# encoder_dim=SATT_ENCODER_DIM reuses the same ResNet-18 spatial features
# as Model 2 so the visual input is identical across all three models
tf_model = TransformerCaptioner(
    vocab_size=VOCAB_SIZE,
    encoder_dim=SATT_ENCODER_DIM,
    d_model=TF_D_MODEL,
    nhead=TF_NHEAD,
    num_layers=TF_LAYERS,
    dim_ff=TF_DIM_FF,
    dropout=TF_DROPOUT,
    freeze_encoder=SATT_FREEZE,
).to(DEVICE)

# ── 3. Optimizer ──────────────────────────────────────────────────────────────
# filter(lambda p: p.requires_grad, ...) excludes the frozen ResNet weights
# so Adam does not allocate momentum buffers for parameters that never update
tf_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, tf_model.parameters()),
    lr=TF_LR, weight_decay=TF_WD,
)

# ── 4. Scheduler ──────────────────────────────────────────────────────────────
tf_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    tf_optimizer, patience=TF_SCHEDULER_PATIENCE, factor=TF_SCHEDULER_FACTOR
)

# ── 5. Sanity Check ───────────────────────────────────────────────────────────
# The parameter count should be comparable to Model 2 because both use the
# same encoder. The decoder parameters differ because self-attention and
# feedforward layers replace the LSTM cell and attention module.
print(f'Trainable parameters: {sum(p.numel() for p in tf_model.parameters() if p.requires_grad):,}')

# ── 6. Training ───────────────────────────────────────────────────────────────
# is_attention is not passed here because TransformerCaptioner.generate
# returns only tokens, not a tuple with attention weights
tf_losses, tf_ppls = train_model(
    tf_model, train_loader, test_loader,
    tf_optimizer, tf_scheduler, DEVICE,
    num_epochs=TF_EPOCHS, model_name='transformer',
    config={"model": "transformer", "lr": TF_LR, "weight_decay": TF_WD,
            "d_model": TF_D_MODEL, "nhead": TF_NHEAD, "num_layers": TF_LAYERS,
            "dim_ff": TF_DIM_FF, "dropout": TF_DROPOUT, "epochs": TF_EPOCHS,
            "scheduler_patience": TF_SCHEDULER_PATIENCE,
            "scheduler_factor": TF_SCHEDULER_FACTOR},
)

plot_curves(tf_losses, tf_ppls, 'Transformer Decoder')

## Final Comparison of All Models

1. **Load Checkpoint** → Restore the best Transformer weights
2. **Evaluate** → Compute perplexity and BLEU scores for the Transformer
3. **Compare** → Print all five models side by side on every metric
4. **Qualitative Evaluation** → Visually inspect Transformer generated captions

---
*Core idea: Bringing all models into one table reveals the full picture of how architecture complexity trades off against caption quality across every metric.*

In [ ]:
# ── 1. Load Checkpoint ────────────────────────────────────────────────────────
tf_model.load_state_dict(torch.load('best_transformer.pth', map_location=DEVICE))

# ── 2. Evaluate ───────────────────────────────────────────────────────────────
# is_attention is not passed because TransformerCaptioner.generate returns
# only tokens and not a tuple with attention weights
tf_ppl  = eval_perplexity(tf_model, test_loader, DEVICE)
tf_bleu = eval_bleu(tf_model, test_loader, tokenizer, DEVICE)

# ── 3. Compare ────────────────────────────────────────────────────────────────
# This table summarizes the full progression from the simplest model to the most
# complex. A higher BLEU-4 score suggests better multi-word coherence while a
# lower perplexity suggests the model assigns higher probability to correct words.
# Note that BLEU and perplexity can disagree because perplexity measures token
# likelihood while BLEU measures surface overlap with reference captions.
print('\n--- All Models Comparison ---')
print(f'{"Metric":<12} {"S&T":>10} {"SAT":>10} {"GloVe":>10} {"Dot-Att":>10} {"Transformer":>14}')

# Parameter count gives context for the complexity vs. performance tradeoff:
# more parameters do not always mean better captions, especially on small datasets
sat_params   = sum(p.numel() for p in sat_model.parameters())
satt_params  = sum(p.numel() for p in satt_model.parameters())
glove_params = sum(p.numel() for p in glove_model.parameters())
dot_params   = sum(p.numel() for p in dot_model.parameters())
tf_params    = sum(p.numel() for p in tf_model.parameters())

print(f'{"Params (M)":<12} {sat_params/1e6:>10.2f} {satt_params/1e6:>10.2f} '
      f'{glove_params/1e6:>10.2f} {dot_params/1e6:>10.2f} {tf_params/1e6:>14.2f}')
print(f'{"Perplexity":<12} {sat_ppl:>10.2f} {satt_ppl:>10.2f} {glove_ppl:>10.2f} {dot_ppl:>10.2f} {tf_ppl:>14.2f}')

# BLEU-1 to BLEU-4 and METEOR are iterated automatically from the eval_bleu dict
for k in sat_bleu:
    print(f'{k:<12} {sat_bleu[k]:>10.4f} {satt_bleu[k]:>10.4f} {glove_bleu[k]:>10.4f} {dot_bleu[k]:>10.4f} {tf_bleu[k]:>14.4f}')

# ── 4. Qualitative Evaluation ─────────────────────────────────────────────────
# Comparing Transformer captions visually against the LSTM captions from earlier
# reveals whether the self-attention mechanism produces more specific or more
# coherent descriptions beyond what the BLEU numbers alone can tell you
show_generated_captions(tf_model, test_loader, tokenizer, DEVICE,
                        n=6, title='Transformer Decoder Generated Captions')

Show and Tell achieves the best perplexity across all models (14.71). The
Transformer reaches 15.27, better than SAT only and worse than S&T,
Dot-Product attention, and GloVe, partially confirming the hypothesis: it
outperforms the attention-based LSTM on perplexity but not the simpler
baselines.

**Final Comparison: All Models**

| Metric     | S&T       | SAT        | GloVe      | Dot-Att    | Transformer |
|------------|-----------|------------|------------|------------|-------------|
| Params (M) | 15.32     | 16.63      | **14.58**  | 16.63      | 14.77       |
| Perplexity | **14.71** | 16.49      | 15.24      | 15.21      | 15.27       |
| BLEU-1     | 0.5952    | **0.6206** | 0.5761     | 0.6018     | 0.5735      |
| BLEU-2     | 0.4156    | **0.4327** | 0.4019     | 0.4229     | 0.3969      |
| BLEU-3     | 0.2782    | **0.2903** | 0.2682     | 0.2861     | 0.2683      |
| BLEU-4     | 0.1863    | **0.1915** | 0.1768     | 0.1902     | 0.1800      |
| METEOR     | 0.3877    | 0a.3767     | 0.3897     | **0.3961** | 0.3877      |

The Transformer reached its best validation-proxy perplexity at epoch 19 within the 20-epoch budget. Since training loss was still decreasing and no separate fine-tuning phase was applied, the comparison is not compute-matched and should be interpreted as exploratory. Qualitatively, it produces more lexically
varied captions ("sidewalk", "snowboarder") compared to the LSTM models but
retains the same core failure patterns, particularly on uncommon activities.
SAT dominates on all BLEU metrics. The persistent BLEU–PPL split across
models suggests that lower perplexity alone does not guarantee better
surface-level caption quality.

Two architectural elements are essential: the **causal mask** prevents the
decoder from attending to future tokens during training, avoiding lookahead
cheating. **Positional encoding** injects order information since the
Transformer processes all tokens in parallel and has no built-in notion of
sequence position.

## Weights & Biases Experiment Tracking

All training runs were tracked with Weights & Biases. The logged values include model configurations, training loss and validation-proxy perplexity. The W&B summaries below document the final logged training state of each run. The final evaluation metrics reported in the model comparison tables were computed separately on the fixed test split and are therefore kept separate from the W&B validation-proxy values.

| Run                          | Phase       | Key setting                    | Epochs / last epoch |   LR |   Dropout | W&B train loss | W&B val perplexity |
| ---------------------------- | ----------- | ------------------------------ | ------------------: | ---: | --------: | -------------: | -----------------: |
| show_and_tell             | pretraining | ResNet18 + LSTM                |    last 35 / max 40 | 3e-4 |       0.5 |         2.4769 |            15.4180 |
| show_and_tell_finetuned    | fine-tuning | encoder unfrozen               |    last 10 / max 10 | 1e-5 | inherited |         2.3621 |            15.3761 |
| show_attend_tell           | pretraining | additive attention             |    last 18 / max 40 | 3e-4 |       0.5 |         2.6638 |            17.1446 |
| show_attend_tell_finetuned | fine-tuning | encoder unfrozen               |     last 9 / max 10 | 1e-5 | inherited |         2.4955 |            16.2856 |
| glove_show_and_tell        | pretraining | GloVe-100 embeddings           |    last 40 / max 40 | 3e-4 |       0.5 |         2.5420 |            15.9333 |
| dot_product_attention      | pretraining | scaled dot-product attention   |    last 40 / max 40 | 3e-4 |       0.5 |         2.4200 |            16.1523 |
| transformer                | training    | 2 layers, 4 heads, d_model 256 |    last 20 / max 20 | 1e-4 |       0.1 |         2.3293 |            16.4038 |

The W&B curves were used to monitor convergence, compare model variants and inspect overfitting behaviour during fine-tuning. Because early stopping and checkpointing were used, the final W&B summary value is not always identical to the best checkpoint used for the final evaluation.


## References and external assistance

The required model architectures are based on Vinyals et al. (2015), *Show and Tell: A Neural Image Caption Generator*, and Xu et al. (2016), *Show, Attend and Tell: Neural Image Caption Generation with Visual Attention*. The Transformer extension follows the decoder and cross-attention principles of Vaswani et al. (2017), *Attention Is All You Need*. GloVe embeddings are based on Pennington et al. (2014), *GloVe: Global Vectors for Word Representation*.

AI tools, including Claude and ChatGPT, were used for debugging suggestions, code review, language refinement, and critical feedback. The implementation, experiments, result interpretation, and submitted content were created, reviewed, adapted, and verified by the author.

